In [ ]:
# =========================================================
# SETUP — 08.01 Training / Checkpoint Audit
#
# Purpose:
#   Inspect the current Fold-0 baseline checkpoint and the
#   local training script before launching any new training.
#
# Questions:
#   1. What training state is stored in the checkpoint?
#   2. Which epoch produced the current best checkpoint?
#   3. Are optimizer / scheduler states saved?
#   4. Does the training script support resume training?
#   5. What epoch / LR / batch / loss controls are exposed?
#
# Changes files:
#   NO
#
# Runs training:
#   NO
#
# Keep after running:
#   YES — documents the starting point for Stage 08.
# =========================================================

from pathlib import Path
import sys
import ast

import torch
import pandas as pd


# ---------------------------------------------------------
# Paths
# ---------------------------------------------------------

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

TRAIN_SCRIPT = (
    PROJECT
    / "scripts"
    / "train_unet_transformer_mps.py"
)

CKPT = (
    PROJECT
    / "models"
    / "official_baseline"
    / "official_baseline_fold0_best.pth"
)

assert TRAIN_SCRIPT.exists()
assert CKPT.exists()


# =========================================================
# 1. Checkpoint structure
# =========================================================

checkpoint = torch.load(
    CKPT,
    map_location="cpu",
    weights_only=False,
)

print("=== CHECKPOINT TYPE ===")
print(type(checkpoint))


if isinstance(checkpoint, dict):

    print("\n=== CHECKPOINT KEYS ===")

    for key in checkpoint.keys():
        value = checkpoint[key]

        if isinstance(
            value,
            (
                int,
                float,
                str,
                bool,
                type(None),
            ),
        ):
            print(
                f"{key}: "
                f"{value!r}"
            )

        elif isinstance(value, dict):
            print(
                f"{key}: dict "
                f"({len(value)} keys)"
            )

        else:
            print(
                f"{key}: "
                f"{type(value).__name__}"
            )


# =========================================================
# 2. Search checkpoint for training-state-like fields
# =========================================================

SEARCH_TERMS = [
    "epoch",
    "step",
    "optimizer",
    "scheduler",
    "lr",
    "loss",
    "acc",
    "recall",
    "score",
    "best",
]

training_state_rows = []

if isinstance(checkpoint, dict):

    for key, value in checkpoint.items():

        if any(
            term in key.lower()
            for term in SEARCH_TERMS
        ):

            if isinstance(
                value,
                (
                    int,
                    float,
                    str,
                    bool,
                    type(None),
                ),
            ):
                description = repr(value)

            elif isinstance(value, dict):
                description = (
                    f"dict ({len(value)} keys)"
                )

            else:
                description = (
                    type(value).__name__
                )

            training_state_rows.append({
                "key":
                    key,

                "value_or_type":
                    description,
            })


print(
    "\n=== CHECKPOINT TRAINING-STATE FIELDS ==="
)

display(
    pd.DataFrame(
        training_state_rows
    )
)


# =========================================================
# 3. Parse training script CLI without running it
# =========================================================

source = TRAIN_SCRIPT.read_text(
    encoding="utf-8"
)

tree = ast.parse(source)

cli_rows = []

for node in ast.walk(tree):

    if not isinstance(node, ast.Call):
        continue

    func = node.func

    is_add_argument = (
        isinstance(func, ast.Attribute)
        and func.attr == "add_argument"
    )

    if not is_add_argument:
        continue

    option_names = []

    for arg in node.args:

        if (
            isinstance(arg, ast.Constant)
            and isinstance(arg.value, str)
        ):
            option_names.append(
                arg.value
            )

    if not option_names:
        continue

    option_text = " / ".join(
        option_names
    )

    relevant_terms = [
        "epoch",
        "resume",
        "weight",
        "checkpoint",
        "batch",
        "learning",
        "lr",
        "seed",
        "split",
        "negative",
        "loss",
    ]

    if not any(
        term in option_text.lower()
        for term in relevant_terms
    ):
        continue

    default_value = None

    for kw in node.keywords:

        if kw.arg == "default":

            try:
                default_value = (
                    ast.literal_eval(
                        kw.value
                    )
                )
            except Exception:
                default_value = (
                    ast.unparse(
                        kw.value
                    )
                )

    cli_rows.append({
        "option":
            option_text,

        "default":
            default_value,

        "line":
            node.lineno,
    })


cli_df = (
    pd.DataFrame(
        cli_rows
    )
    .sort_values(
        "line"
    )
    .reset_index(
        drop=True
    )
)


print(
    "\n=== RELEVANT TRAINING CLI OPTIONS ==="
)

display(
    cli_df
)


# =========================================================
# 4. Explicit resume-support search
# =========================================================

resume_lines = []

for i, line in enumerate(
    source.splitlines(),
    start=1,
):

    lower = line.lower()

    if (
        "resume" in lower
        or "optimizer.load_state_dict" in lower
        or "scheduler.load_state_dict" in lower
    ):

        resume_lines.append({
            "line":
                i,

            "code":
                line.strip(),
        })


print(
    "\n=== RESUME-TRAINING EVIDENCE ==="
)

if resume_lines:
    display(
        pd.DataFrame(
            resume_lines
        )
    )
else:
    print(
        "No explicit resume-training logic found."
    )


print(
    "\n08.01 training/checkpoint audit: OK"
)

In [ ]:
# =========================================================
# SETUP — 08.02 Training Initialization & Checkpoint Logic Audit
#
# Purpose:
#   Inspect the exact training initialization and checkpoint
#   selection/saving logic before launching a longer run.
#
# Questions:
#   1. What exactly does --unet-weights load?
#   2. Can the current full-model checkpoint be warm-started
#      without modifying the training script?
#   3. What metric selects the "best" checkpoint?
#   4. What exactly is written to the .pth file?
#
# Changes files:
#   NO
#
# Runs training:
#   NO
#
# Keep after running:
#   YES — formal Stage 08 training audit.
# =========================================================

from pathlib import Path
import re

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

TRAIN_SCRIPT = (
    PROJECT
    / "scripts"
    / "train_unet_transformer_mps.py"
)

assert TRAIN_SCRIPT.exists()

lines = TRAIN_SCRIPT.read_text(
    encoding="utf-8"
).splitlines()


# ---------------------------------------------------------
# Search targets
# ---------------------------------------------------------

PATTERNS = {
    "unet_weights": [
        r"unet_weights",
        r"unet-weights",
    ],

    "checkpoint_load": [
        r"torch\.load",
        r"load_state_dict",
    ],

    "checkpoint_save": [
        r"torch\.save",
        r"state_dict\(",
    ],

    "best_selection": [
        r"\bbest\b",
        r"test_acc",
        r"test_recall",
        r"val_acc",
        r"val_recall",
    ],

    "epoch_loop": [
        r"for .*epoch",
        r"range\(.*epochs",
    ],
}


def collect_hits(patterns):
    hits = []

    for i, line in enumerate(
        lines,
        start=1,
    ):
        if any(
            re.search(
                pattern,
                line,
                flags=re.IGNORECASE,
            )
            for pattern in patterns
        ):
            hits.append(i)

    return hits


def merge_blocks(
    hit_lines,
    before=8,
    after=12,
):
    blocks = []

    for line_no in hit_lines:

        start = max(
            1,
            line_no - before,
        )

        end = min(
            len(lines),
            line_no + after,
        )

        if (
            blocks
            and start <= blocks[-1][1] + 1
        ):
            blocks[-1] = (
                blocks[-1][0],
                max(
                    blocks[-1][1],
                    end,
                ),
            )
        else:
            blocks.append(
                (start, end)
            )

    return blocks


for section, patterns in PATTERNS.items():

    hits = collect_hits(
        patterns
    )

    print(
        "\n"
        + "=" * 80
    )

    print(
        f"=== {section.upper()} ==="
    )

    print(
        "Hit lines:",
        hits,
    )

    print(
        "=" * 80
    )

    blocks = merge_blocks(
        hits,
        before=8,
        after=12,
    )

    for start, end in blocks:

        print(
            f"\n--- lines {start}-{end} ---"
        )

        for n in range(
            start,
            end + 1,
        ):
            print(
                f"{n:5d}: {lines[n - 1]}"
            )


print(
    "\n08.02 training initialization/checkpoint audit: OK"
)

In [ ]:
# =========================================================
# SETUP — 08.05 Baseline Training Configuration Lock
#
# Purpose:
#   Lock the exact training-side defaults/settings that
#   affect a warm-start continuation experiment.
#
# We specifically verify:
#   - epochs
#   - learning rate
#   - batch size
#   - detection loss weight
#   - negative voxel weight
#   - pool kernel
#   - window size
#   - downsample
#   - augmentations
#
# Changes files:
#   NO
#
# Runs training:
#   NO
#
# Keep after running:
#   YES — formal configuration record.
# =========================================================

from pathlib import Path
import ast
import pandas as pd


PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

TRAIN_SCRIPT = (
    PROJECT
    / "scripts"
    / "train_unet_transformer_mps.py"
)

source = TRAIN_SCRIPT.read_text(
    encoding="utf-8"
)

tree = ast.parse(source)


# =========================================================
# 1. Extract train() signature defaults
# =========================================================

train_defaults = []

for node in tree.body:

    if (
        isinstance(node, ast.FunctionDef)
        and node.name == "train"
    ):

        args = node.args.args

        defaults = node.args.defaults

        n_without_defaults = (
            len(args) - len(defaults)
        )

        for i, arg in enumerate(args):

            if i < n_without_defaults:
                default_value = "<required>"

            else:

                default_node = defaults[
                    i - n_without_defaults
                ]

                try:
                    default_value = ast.literal_eval(
                        default_node
                    )
                except Exception:
                    default_value = ast.unparse(
                        default_node
                    )

            train_defaults.append({
                "parameter":
                    arg.arg,

                "default":
                    default_value,
            })

        break


train_defaults_df = pd.DataFrame(
    train_defaults
)


INTERESTING_PARAMS = [
    "n_epochs",
    "lr",
    "batch_size",
    "unet_out_channels",
    "unet_layers",
    "downsample",
    "det_loss_weight",
    "det_neg_weight",
    "max_iters",
    "seed",
    "max_frames",
    "window_size",
    "augmentations",
    "pool_kernel_um",
]


print(
    "=== TRAIN() DEFAULTS ==="
)

display(
    train_defaults_df[
        train_defaults_df[
            "parameter"
        ].isin(
            INTERESTING_PARAMS
        )
    ]
)


# =========================================================
# 2. Extract relevant CLI defaults
# =========================================================

CLI_NAMES = [
    "--epochs",
    "--lr",
    "--batch-size",
    "--unet-out-channels",
    "--unet-layers",
    "--downsample",
    "--det-loss-weight",
    "--det-neg-weight",
    "--max-iters",
    "--max-frames",
    "--seed",
    "--window-size",
    "--pool-kernel-um",
    "--single-gpu",
]


cli_rows = []

for node in ast.walk(tree):

    if not isinstance(node, ast.Call):
        continue

    if not (
        isinstance(node.func, ast.Attribute)
        and node.func.attr == "add_argument"
    ):
        continue

    option_names = []

    for arg in node.args:

        if (
            isinstance(arg, ast.Constant)
            and isinstance(arg.value, str)
        ):
            option_names.append(
                arg.value
            )

    if not any(
        option in CLI_NAMES
        for option in option_names
    ):
        continue

    default = "<argparse implicit>"

    for kw in node.keywords:

        if kw.arg == "default":

            try:
                default = ast.literal_eval(
                    kw.value
                )
            except Exception:
                default = ast.unparse(
                    kw.value
                )

    cli_rows.append({
        "option":
            " / ".join(option_names),

        "default":
            default,

        "line":
            node.lineno,
    })


cli_defaults_df = (
    pd.DataFrame(cli_rows)
    .sort_values("line")
    .reset_index(drop=True)
)


print(
    "\n=== TRAINING CLI DEFAULTS ==="
)

display(
    cli_defaults_df
)


# =========================================================
# 3. Explicitly print the warm-start experiment proposal
# =========================================================

proposed_warmstart = pd.DataFrame([
    {
        "setting": "initial_weights",
        "value": "official_baseline_fold0_best.pth",
    },
    {
        "setting": "additional_epochs",
        "value": 1,
    },
    {
        "setting": "optimizer",
        "value": "fresh AdamW",
    },
    {
        "setting": "learning_rate",
        "value": 1e-5,
    },
    {
        "setting": "det_loss_weight",
        "value": 1.0,
    },
    {
        "setting": "det_neg_weight",
        "value": 0.01,
    },
    {
        "setting": "seed",
        "value": 1640,
    },
])


print(
    "\n=== PROPOSED FIRST WARM-START EXPERIMENT ==="
)

display(
    proposed_warmstart
)


print(
    "\n08.05 baseline training configuration lock: OK"
)

## 08.06 First Full-Model Warm-Start Training

**Status:** ready for Kaggle GPU run

Frozen configuration:

- Initial weights: `official_baseline_fold0_best.pth`
- Additional epochs: `1`
- Fresh AdamW optimizer
- Learning rate: `1e-5`
- Batch size: `8`
- Detection loss weight: `1.0`
- Detection negative weight: `0.01`
- Window size: `2`
- Downsample: `(1,4,4)`
- Pool kernel: `5.0 µm` → realized `(3,3,3)`
- Seed: `1640`
- Fold: `0`

This is a one-epoch continuation experiment. The resulting checkpoint will be evaluated with the locked official competition metric and baseline-v4 post-processing before any further training is attempted.

## 08.06 — Full-Model Warm-Start Training Result

**Status:** COMPLETE

- Initial checkpoint: `official_baseline_fold0_best.pth`
- Initial SHA256: `4a3943a2a4d7a8e288851b5736972d84793e29f88fe334c124597da288fbb600`
- Additional epochs: `1`
- Optimizer: fresh AdamW
- Learning rate: `1e-5`
- Batch size: `8`
- Detection loss weight: `1.0`
- Detection negative weight: `0.01`
- Window size: `2`
- Pool kernel: `5.0 µm`
- Seed: `1640`
- Device: MPS
- Train windows: `14,919`
- Validation windows: `3,788`
- Training batches: `1,865`
- Validation recall: `0.9103`
- Internal `acc × recall`: `0.9100`
- Runtime: ~4 h 49 min

**Output checkpoint**

`models/official_baseline/official_baseline_fold0_warm1_lr1e5/split_0/edge_predictor_best.pth`

SHA256:

`018ec5d91104a880b61351e5bbe87ef9e2fb102bc69d14df17b6ae95e6ca0b00`

Original frozen checkpoint remained unchanged.

**Interpretation:** internal training metrics are not the competition metric. The checkpoint must be evaluated using baseline-v4 + locked official metric before deciding whether further training is useful.

In [ ]:
# ============================================================
# SETUP — 08.07 Warm-Start Inference Interface Audit
# Purpose:
#   Inspect the exact CLI of the frozen local prediction adapter
#   before running full Fold0 inference for the new warm-start
#   checkpoint. Adds the locked official scripts directory to
#   the subprocess PYTHONPATH so baseline imports resolve.
#
# Changes files: NO
# Runs training: NO
# Runs inference: NO
# Uses GT: NO
# Keep after running: YES
# ============================================================

from pathlib import Path
import subprocess
import sys
import os
import hashlib

PROJECT_ROOT = Path("/Users/heqiuyan/Desktop/biohub-cell-tracking")

PREDICT_SCRIPT = (
    PROJECT_ROOT
    / "scripts/predict_unet_transformer_mps.py"
)

OFFICIAL_SCRIPTS = (
    PROJECT_ROOT
    / "external/official/scripts"
)

WARM_CHECKPOINT = (
    PROJECT_ROOT
    / "models/official_baseline"
    / "official_baseline_fold0_warm1_lr1e5"
    / "split_0"
    / "edge_predictor_best.pth"
)


def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


# ------------------------------------------------------------
# Pre-flight
# ------------------------------------------------------------

assert PREDICT_SCRIPT.exists(), PREDICT_SCRIPT
assert OFFICIAL_SCRIPTS.exists(), OFFICIAL_SCRIPTS
assert (OFFICIAL_SCRIPTS / "augmentations.py").exists()
assert WARM_CHECKPOINT.exists(), WARM_CHECKPOINT


# ------------------------------------------------------------
# Subprocess environment
# ------------------------------------------------------------

env = os.environ.copy()

existing_pythonpath = env.get("PYTHONPATH", "")

env["PYTHONPATH"] = (
    str(OFFICIAL_SCRIPTS)
    if not existing_pythonpath
    else str(OFFICIAL_SCRIPTS) + os.pathsep + existing_pythonpath
)


# ------------------------------------------------------------
# Audit
# ------------------------------------------------------------

print("=== 08.07 WARM-START INFERENCE INTERFACE AUDIT ===")
print()
print(f"Prediction script: {PREDICT_SCRIPT}")
print(f"Script SHA256:     {sha256(PREDICT_SCRIPT)}")
print()
print(f"Warm checkpoint:   {WARM_CHECKPOINT}")
print(f"Checkpoint SHA256: {sha256(WARM_CHECKPOINT)}")
print()
print(f"PYTHONPATH:        {env['PYTHONPATH']}")
print()

result = subprocess.run(
    [sys.executable, str(PREDICT_SCRIPT), "--help"],
    cwd=PROJECT_ROOT,
    env=env,
    capture_output=True,
    text=True,
    check=False,
)

print("=" * 80)
print("RETURN CODE")
print("=" * 80)
print(result.returncode)

print("\n" + "=" * 80)
print("PREDICTION CLI")
print("=" * 80)
print(result.stdout)

if result.stderr.strip():
    print("\n" + "=" * 80)
    print("STDERR")
    print("=" * 80)
    print(result.stderr)

assert result.returncode == 0, (
    "Prediction CLI audit failed. Inspect STDERR above."
)

print("\n08.07 inference interface audit: COMPLETE")

## 08.08 — Warm-Start Dual-Threshold Inference Result

**Status:** COMPLETE

Checkpoint:
`official_baseline_fold0_warm1_lr1e5/split_0/edge_predictor_best.pth`

SHA256:
`018ec5d91104a880b61351e5bbe87ef9e2fb102bc69d14df17b6ae95e6ca0b00`

Fold0 full inference:

- `det_threshold = 0.995`: 40/40 GEFF, 37.2 min
- `det_threshold = 0.9975`: 40/40 GEFF, 37.1 min

Output coverage: **OK**

Detailed logs:
`reports/inference_logs/`

Next: apply the frozen baseline-v4 post-processing and evaluate with the locked official metric.

In [ ]:
# ============================================================
# RUN — 08.12 Warm-Start Baseline-v4 Official Evaluation
# Purpose:
#   Evaluate the warm-start Fold0 predictions using the exact
#   frozen baseline-v4 policy and locked official metric.
#
#   First reproduce the original baseline-v4 score as a gate,
#   then evaluate the warm-start model under identical rules.
#
# Changes files: NO
# Runs training: NO
# Runs inference: NO
# Uses GT: YES — official Fold0 evaluation only
# Keep after running: NO — replace with RESULT markdown
# ============================================================

from pathlib import Path
from collections import defaultdict
import sys
import warnings

import numpy as np
import pandas as pd
import tracksdata as td


# ============================================================
# Paths
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

TRAIN_DIR = (
    PROJECT
    / "data/raw/competition/"
      "biohub-cell-tracking-during-development/train"
)

CV_PATH = (
    PROJECT
    / "configs/cv_manifest.csv"
)

OFFICIAL_SRC = (
    PROJECT
    / "external/official/src"
)

OFFICIAL_SCRIPTS = (
    PROJECT
    / "external/official/scripts"
)

for p in [OFFICIAL_SRC, OFFICIAL_SCRIPTS]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))


# ============================================================
# Locked official metric
# ============================================================

from tracking_cellmot.io import open_dataset
from tracking_cellmot.metrics import (
    evaluate as official_evaluate,
    node_recall,
    per_sample_metrics,
    summarise,
)

warnings.filterwarnings(
    "ignore",
    message="Graph already matched, overwriting previous matching.",
)


# ============================================================
# Frozen baseline-v4 configuration
# ============================================================

G2_P2_MIN = 0.55
G2_DAUGHTER_SEP_MIN_UM = 6.0
G2_ANGLE_MIN_DEG = 120.0
G2_MAX_PARENT_DAUGHTER_UM = 8.0
G2_DISTANCE_BALANCE_MAX = 0.50

TWO_NODE_MAX_DIST_UM = 3.5
CONF_COMPONENT_LIMIT = 4

FROZEN_BASELINE_V4_SCORE = 0.744252


# ============================================================
# CV metadata
# ============================================================

cv_meta = pd.read_csv(CV_PATH)

fold0 = (
    cv_meta[
        cv_meta["fold"] == 0
    ]
    .copy()
)

assert len(fold0) == 40

prefix_map = dict(
    zip(
        fold0["dataset"],
        fold0["prefix"],
    )
)

estimated_map = dict(
    zip(
        fold0["dataset"],
        fold0["estimated_nodes"],
    )
)

dataset_names = sorted(
    fold0["dataset"]
)


# ============================================================
# Locate prediction directories robustly
# ============================================================

PREDICTIONS_ROOT = PROJECT / "predictions"


def find_prediction_dir(method):
    candidates = []

    for split_dir in PREDICTIONS_ROOT.rglob("split_0"):
        if split_dir.parent.name != method:
            continue

        geffs = list(
            split_dir.glob("*.geff")
        )

        if len(geffs) == 40:
            candidates.append(
                split_dir
            )

    assert len(candidates) == 1, (
        f"Expected exactly one 40-GEFF directory for:\n"
        f"{method}\n"
        f"Found: {candidates}"
    )

    return candidates[0]


# Original frozen predictions
OLD_0995_DIR = find_prediction_dir(
    "official_baseline_fold0_det09950"
)

OLD_09975_DIR = find_prediction_dir(
    "official_baseline_fold0_det09975"
)

# Warm-start predictions
WARM_0995_DIR = find_prediction_dir(
    "official_baseline_fold0_warm1_lr1e5_det0995"
)

WARM_09975_DIR = find_prediction_dir(
    "official_baseline_fold0_warm1_lr1e5_det09975"
)


# ============================================================
# Graph helpers
# ============================================================

def load_graph(path):
    loaded = (
        td.graph.IndexedRXGraph
        .from_geff(path)
    )

    return (
        loaded[0]
        if isinstance(loaded, tuple)
        else loaded
    )


def node_coordinate_keys(graph):
    df = (
        graph
        .node_attrs(unpack=True)
        .to_pandas()
    )

    return {
        (
            int(row.t),
            int(round(row.z)),
            int(round(row.y)),
            int(round(row.x)),
        )
        for row in df.itertuples(
            index=False
        )
    }


def component_sets(graph):
    node_ids = [
        int(x)
        for x in graph
        .node_attrs(unpack=True)[
            "node_id"
        ]
        .to_list()
    ]

    parent = {
        n: n
        for n in node_ids
    }

    size = {
        n: 1
        for n in node_ids
    }

    def find(x):
        root = x

        while parent[root] != root:
            root = parent[root]

        while parent[x] != x:
            nxt = parent[x]
            parent[x] = root
            x = nxt

        return root

    def union(a, b):
        ra = find(a)
        rb = find(b)

        if ra == rb:
            return

        if size[ra] < size[rb]:
            ra, rb = rb, ra

        parent[rb] = ra
        size[ra] += size[rb]

    for source, target in graph.edge_list():
        union(
            int(source),
            int(target),
        )

    components = defaultdict(list)

    for node_id in node_ids:
        components[
            find(node_id)
        ].append(node_id)

    return list(
        components.values()
    )


# ============================================================
# Frozen G2
# ============================================================

def apply_frozen_g2(
    graph,
    scale,
):
    scale = np.asarray(
        scale,
        dtype=float,
    )

    nodes = (
        graph
        .node_attrs(unpack=True)
        .to_pandas()
        .set_index("node_id")
    )

    edges = (
        graph
        .edge_attrs(
            attr_keys=["edge_prob"],
            unpack=True,
        )
        .to_pandas()
    )

    remove_ids = []

    for source_id, group in edges.groupby(
        "source_id"
    ):
        if len(group) != 2:
            continue

        group = group.sort_values(
            ["edge_prob", "edge_id"],
            ascending=[False, True],
        )

        e1 = group.iloc[0]
        e2 = group.iloc[1]

        source_id = int(source_id)
        target1 = int(e1["target_id"])
        target2 = int(e2["target_id"])

        parent = nodes.loc[source_id]
        child1 = nodes.loc[target1]
        child2 = nodes.loc[target2]

        p = (
            np.array(
                [
                    parent["z"],
                    parent["y"],
                    parent["x"],
                ],
                dtype=float,
            )
            * scale
        )

        c1 = (
            np.array(
                [
                    child1["z"],
                    child1["y"],
                    child1["x"],
                ],
                dtype=float,
            )
            * scale
        )

        c2 = (
            np.array(
                [
                    child2["z"],
                    child2["y"],
                    child2["x"],
                ],
                dtype=float,
            )
            * scale
        )

        v1 = c1 - p
        v2 = c2 - p

        d1 = float(
            np.linalg.norm(v1)
        )

        d2 = float(
            np.linalg.norm(v2)
        )

        daughter_sep = float(
            np.linalg.norm(
                c1 - c2
            )
        )

        if d1 > 0 and d2 > 0:
            cosine = float(
                np.dot(v1, v2)
                / (d1 * d2)
            )

            cosine = float(
                np.clip(
                    cosine,
                    -1.0,
                    1.0,
                )
            )

            angle = float(
                np.degrees(
                    np.arccos(
                        cosine
                    )
                )
            )

        else:
            angle = np.nan

        distance_balance = (
            abs(d1 - d2)
            / max(
                d1 + d2,
                1e-12,
            )
        )

        keep_second = (
            float(e2["edge_prob"])
            >= G2_P2_MIN

            and daughter_sep
            >= G2_DAUGHTER_SEP_MIN_UM

            and np.isfinite(angle)

            and angle
            >= G2_ANGLE_MIN_DEG

            and max(d1, d2)
            <= G2_MAX_PARENT_DAUGHTER_UM

            and distance_balance
            <= G2_DISTANCE_BALANCE_MAX
        )

        if not keep_second:
            remove_ids.append(
                int(e2["edge_id"])
            )

    if remove_ids:
        graph.bulk_remove_edges(
            remove_ids
        )


# ============================================================
# Frozen confidence-aware K<=4 pruning
# ============================================================

def apply_low_conf_component_pruning(
    graph,
    high_conf_coordinate_set,
):
    nodes = (
        graph
        .node_attrs(unpack=True)
        .to_pandas()
        .set_index("node_id")
    )

    comps = component_sets(
        graph
    )

    remove_nodes = set()

    for comp in comps:
        comp = list(
            map(int, comp)
        )

        # Frozen K <= 4 rule
        if len(comp) > CONF_COMPONENT_LIMIT:
            continue

        for node_id in comp:
            row = nodes.loc[
                node_id
            ]

            key = (
                int(row["t"]),
                int(round(row["z"])),
                int(round(row["y"])),
                int(round(row["x"])),
            )

            if key not in high_conf_coordinate_set:
                remove_nodes.add(
                    node_id
                )

    if remove_nodes:
        graph.bulk_remove_nodes(
            sorted(remove_nodes)
        )

    return len(remove_nodes)


# ============================================================
# Frozen component cleanup
# ============================================================

def apply_frozen_component_cleanup(
    graph,
    scale,
):
    scale = np.asarray(
        scale,
        dtype=float,
    )

    nodes = (
        graph
        .node_attrs(unpack=True)
        .to_pandas()
        .set_index("node_id")
    )

    edges = (
        graph
        .edge_attrs(
            attr_keys=["edge_prob"],
            unpack=True,
        )
        .to_pandas()
    )

    edge_lookup = {}

    for _, e in edges.iterrows():
        s = int(e["source_id"])
        t = int(e["target_id"])

        edge_lookup[
            frozenset([s, t])
        ] = (s, t)

    remove_nodes = set()

    for comp in component_sets(graph):
        comp = list(
            map(int, comp)
        )

        # Singleton
        if len(comp) == 1:
            remove_nodes.add(
                comp[0]
            )

        # 2-node component
        elif len(comp) == 2:
            key = frozenset(comp)

            if key not in edge_lookup:
                continue

            source, target = (
                edge_lookup[key]
            )

            a = nodes.loc[source]
            b = nodes.loc[target]

            delta_um = (
                np.array(
                    [
                        float(
                            b["z"] - a["z"]
                        ),
                        float(
                            b["y"] - a["y"]
                        ),
                        float(
                            b["x"] - a["x"]
                        ),
                    ]
                )
                * scale
            )

            dist_um = float(
                np.linalg.norm(
                    delta_um
                )
            )

            if (
                dist_um
                > TWO_NODE_MAX_DIST_UM
            ):
                remove_nodes.update(
                    comp
                )

    if remove_nodes:
        graph.bulk_remove_nodes(
            sorted(remove_nodes)
        )

    return len(remove_nodes)


# ============================================================
# Evaluation
# ============================================================

def evaluate_baseline_v4(
    pred_0995_dir,
    pred_09975_dir,
):
    records = []
    rows = []

    nesting_violations = 0
    conf_removed_total = 0
    cleanup_removed_total = 0

    for name in dataset_names:

        main_path = (
            pred_0995_dir
            / f"{name}.geff"
        )

        high_path = (
            pred_09975_dir
            / f"{name}.geff"
        )

        gt_path = (
            TRAIN_DIR
            / f"{name}.geff"
        )

        assert main_path.exists()
        assert high_path.exists()
        assert gt_path.exists()

        pred_g = load_graph(
            main_path
        )

        high_g = load_graph(
            high_path
        )

        gt_g = load_graph(
            gt_path
        )

        # ----------------------------------------
        # Threshold-survival confidence
        # ----------------------------------------

        main_keys = (
            node_coordinate_keys(
                pred_g
            )
        )

        high_keys = (
            node_coordinate_keys(
                high_g
            )
        )

        violations = (
            high_keys
            - main_keys
        )

        nesting_violations += len(
            violations
        )

        assert not violations, (
            f"Threshold nesting violation "
            f"in {name}: {len(violations)}"
        )

        # ----------------------------------------
        # Physical scale
        # ----------------------------------------

        scale = np.asarray(
            open_dataset(
                TRAIN_DIR / name,
                load_image=False,
            ).scale,
            dtype=float,
        )

        # ----------------------------------------
        # 1. Frozen G2
        # ----------------------------------------

        apply_frozen_g2(
            pred_g,
            scale,
        )

        # ----------------------------------------
        # 2. Frozen confidence K<=4 pruning
        # ----------------------------------------

        conf_removed_total += (
            apply_low_conf_component_pruning(
                pred_g,
                high_keys,
            )
        )

        # ----------------------------------------
        # 3. Frozen graph cleanup
        # ----------------------------------------

        cleanup_removed_total += (
            apply_frozen_component_cleanup(
                pred_g,
                scale,
            )
        )

        # ----------------------------------------
        # 4. Locked official metric
        # ----------------------------------------

        er = official_evaluate(
            pred_g,
            gt_g,
            scale=tuple(scale),
            max_distance=7.0,
        )

        recall = node_recall(
            pred_g,
            gt_g,
        )

        m = per_sample_metrics(
            er,
            float(
                estimated_map[name]
            ),
            recall,
        )

        records.append(m)

        rows.append({
            "dataset":
                name,

            "prefix":
                prefix_map[name],

            "edge_jaccard":
                m["edge_jaccard"],

            "adj_edge_jaccard":
                m["adj_edge_jaccard"],

            "node_recall":
                m["node_recall"],

            "node_ratio":
                m["total_node_ratio"],
        })

    assert nesting_violations == 0

    s = summarise(
        records
    )

    overall = {
        "edge_jaccard":
            s["edge_jaccard"],

        "adj_edge_jaccard":
            s["adj_edge_jaccard"],

        "division_jaccard":
            s["division_jaccard"],

        "node_recall":
            s["node_recall"],

        "median_node_ratio":
            float(
                np.median([
                    m["total_node_ratio"]
                    for m in records
                ])
            ),

        "score":
            s["score"],

        "confidence_nodes_removed":
            conf_removed_total,

        "cleanup_nodes_removed":
            cleanup_removed_total,
    }

    dataset_df = pd.DataFrame(
        rows
    )

    prefix_rows = []

    for prefix in [
        "44b6",
        "6bba",
    ]:
        prefix_names = set(
            dataset_df.loc[
                dataset_df["prefix"]
                == prefix,
                "dataset",
            ]
        )

        prefix_records = [
            m
            for name, m in zip(
                dataset_names,
                records,
            )
            if name in prefix_names
        ]

        ps = summarise(
            prefix_records
        )

        prefix_ratios = (
            dataset_df.loc[
                dataset_df["prefix"]
                == prefix,
                "node_ratio",
            ]
        )

        prefix_rows.append({
            "prefix":
                prefix,

            "edge_jaccard":
                ps["edge_jaccard"],

            "adj_edge_jaccard":
                ps["adj_edge_jaccard"],

            "division_jaccard":
                ps["division_jaccard"],

            "node_recall":
                ps["node_recall"],

            "median_node_ratio":
                float(
                    np.median(
                        prefix_ratios
                    )
                ),

            "score":
                ps["score"],
        })

    return (
        overall,
        pd.DataFrame(prefix_rows),
        dataset_df,
    )


# ============================================================
# Preflight summary
# ============================================================

print("=== 08.12 BASELINE-v4 OFFICIAL EVALUATION ===")
print()
print("Original .995 :", OLD_0995_DIR)
print("Original .9975:", OLD_09975_DIR)
print("Warm .995     :", WARM_0995_DIR)
print("Warm .9975    :", WARM_09975_DIR)
print()
print("Evaluating original reproduction gate...")


# ============================================================
# Original baseline-v4 reproduction
# ============================================================

old_overall, old_prefix, _ = (
    evaluate_baseline_v4(
        OLD_0995_DIR,
        OLD_09975_DIR,
    )
)

old_score = float(
    old_overall["score"]
)

old_delta = (
    old_score
    - FROZEN_BASELINE_V4_SCORE
)

print(
    f"Original reproduced score: "
    f"{old_score:.6f}"
)

print(
    f"Difference vs 0.744252:    "
    f"{old_delta:+.6f}"
)

assert abs(old_delta) < 5e-6, (
    "Frozen baseline-v4 did not reproduce. "
    "Stop before evaluating the warm model."
)


# ============================================================
# Warm-start baseline-v4
# ============================================================

print()
print("Reproduction gate: OK")
print("Evaluating warm-start model...")

warm_overall, warm_prefix, warm_dataset = (
    evaluate_baseline_v4(
        WARM_0995_DIR,
        WARM_09975_DIR,
    )
)


# ============================================================
# Final comparison
# ============================================================

comparison = pd.DataFrame([
    {
        "model": "original_baseline_v4",
        **old_overall,
    },
    {
        "model": "warm1_lr1e5_v4",
        **warm_overall,
    },
])

warm_score = float(
    warm_overall["score"]
)

score_delta = (
    warm_score
    - old_score
)

print()
print("=" * 72)
print("08.12 RESULT")
print("=" * 72)

display(
    comparison[
        [
            "model",
            "edge_jaccard",
            "adj_edge_jaccard",
            "division_jaccard",
            "node_recall",
            "median_node_ratio",
            "score",
        ]
    ]
)

print(
    f"Score delta: "
    f"{score_delta:+.6f}"
)

print()
print("Warm-start by prefix:")

display(
    warm_prefix[
        [
            "prefix",
            "edge_jaccard",
            "adj_edge_jaccard",
            "division_jaccard",
            "node_recall",
            "median_node_ratio",
            "score",
        ]
    ]
)

print()
print(
    "Confidence nodes removed:",
    warm_overall[
        "confidence_nodes_removed"
    ],
)

print(
    "Cleanup nodes removed:",
    warm_overall[
        "cleanup_nodes_removed"
    ],
)

print()
print(
    "08.12 warm-start baseline-v4 "
    "official evaluation: COMPLETE"
)

## 08.12 — Warm-Start Baseline-v4 Evaluation

**Status: SUCCESS — NEW BEST**

Fixed-policy comparison using the locked baseline-v4 post-processing and official metric.

| Metric | Original baseline-v4 | Warm1 lr=1e-5 |
|---|---:|---:|
| Edge Jaccard | 0.741920 | **0.761540** |
| Adjusted Edge Jaccard | 0.741221 | **0.753573** |
| Division Jaccard | **0.030303** | 0.025641 |
| Node Recall | 0.929858 | **0.954890** |
| Median Node Ratio | +0.004419 | +0.072152 |
| Score | 0.744252 | **0.756137** |

**Score improvement: +0.011885**

Prefix scores:

- 44b6: **0.760060**
- 6bba: **0.755224**

Original baseline-v4 reproduction:
`0.744252` — exact reproduction confirmed.

Conclusion:
One additional full-model warm-start epoch with fresh AdamW and
`lr=1e-5` produced a substantial, cross-domain Fold0 improvement.

Current best checkpoint:
`official_baseline_fold0_warm1_lr1e5`

In [ ]:
# ============================================================
# RUN — 08.13 Warm-Start Confidence-K Recalibration
# Purpose:
#   Re-evaluate the confidence-aware component-size cutoff K
#   for the new warm-start checkpoint while keeping every
#   other baseline-v4 rule frozen.
#
# Changes files: NO
# Runs training: NO
# Runs inference: NO
# Uses GT: YES — Fold0 metric selection
# Keep after running: YES — formal post-processing calibration
# ============================================================

CONF_LIMITS = [
    0,
    2,
    4,
    9,
    19,
    np.inf,
]

rows = []

print("=== 08.13 WARM-START K RECALIBRATION ===")
print()

for max_comp_size in CONF_LIMITS:

    records = []
    conf_removed = 0
    cleanup_removed = 0

    for name in dataset_names:

        pred_g = load_graph(
            WARM_0995_DIR / f"{name}.geff"
        )

        high_g = load_graph(
            WARM_09975_DIR / f"{name}.geff"
        )

        gt_g = load_graph(
            TRAIN_DIR / f"{name}.geff"
        )

        high_keys = node_coordinate_keys(
            high_g
        )

        scale = np.asarray(
            open_dataset(
                TRAIN_DIR / name,
                load_image=False,
            ).scale,
            dtype=float,
        )

        # 1. Frozen G2
        apply_frozen_g2(
            pred_g,
            scale,
        )

        # 2. Confidence pruning with candidate K
        if max_comp_size != 0:

            nodes = (
                pred_g
                .node_attrs(unpack=True)
                .to_pandas()
                .set_index("node_id")
            )

            remove_nodes = set()

            for comp in component_sets(pred_g):

                comp = list(
                    map(int, comp)
                )

                if (
                    not np.isinf(max_comp_size)
                    and len(comp) > max_comp_size
                ):
                    continue

                for node_id in comp:

                    row = nodes.loc[node_id]

                    key = (
                        int(row["t"]),
                        int(round(row["z"])),
                        int(round(row["y"])),
                        int(round(row["x"])),
                    )

                    if key not in high_keys:
                        remove_nodes.add(
                            node_id
                        )

            if remove_nodes:
                pred_g.bulk_remove_nodes(
                    sorted(remove_nodes)
                )

            conf_removed += len(
                remove_nodes
            )

        # 3. Frozen cleanup
        cleanup_removed += (
            apply_frozen_component_cleanup(
                pred_g,
                scale,
            )
        )

        # 4. Official metric
        er = official_evaluate(
            pred_g,
            gt_g,
            scale=tuple(scale),
            max_distance=7.0,
        )

        recall = node_recall(
            pred_g,
            gt_g,
        )

        m = per_sample_metrics(
            er,
            float(estimated_map[name]),
            recall,
        )

        records.append(m)

    s = summarise(records)

    rows.append({
        "K":
            "all"
            if np.isinf(max_comp_size)
            else int(max_comp_size),

        "edge_jaccard":
            s["edge_jaccard"],

        "adj_edge_jaccard":
            s["adj_edge_jaccard"],

        "division_jaccard":
            s["division_jaccard"],

        "node_recall":
            s["node_recall"],

        "median_node_ratio":
            float(
                np.median([
                    m["total_node_ratio"]
                    for m in records
                ])
            ),

        "score":
            s["score"],

        "conf_nodes_removed":
            conf_removed,

        "cleanup_nodes_removed":
            cleanup_removed,
    })


k_sweep = pd.DataFrame(rows)

print("08.13 RESULT")
display(k_sweep)

best_idx = (
    k_sweep["score"]
    .astype(float)
    .idxmax()
)

best = k_sweep.loc[
    best_idx
]

print()
print(
    "Best K:",
    best["K"],
)

print(
    "Best score:",
    f"{best['score']:.6f}",
)

print(
    "Delta vs fixed K=4 warm baseline:",
    f"{best['score'] - 0.756137:+.6f}",
)

print()
print("08.13 warm-start K recalibration: COMPLETE")

In [ ]:
# ============================================================
# RUN — 08.14 Warm-Start K4 vs K9 Robustness Check
# Purpose:
#   Compare K=4 and K=9 under the warm-start checkpoint by:
#     1. overall metric
#     2. 44b6 / 6bba prefix
#     3. paired per-dataset metric changes
#
# Changes files: NO
# Runs training: NO
# Runs inference: NO
# Uses GT: YES — Fold0 robustness evaluation
# Keep after running: YES — formal calibration evidence
# ============================================================

def evaluate_warm_k(max_comp_size):

    records = []
    dataset_rows = []

    for name in dataset_names:

        pred_g = load_graph(
            WARM_0995_DIR / f"{name}.geff"
        )

        high_g = load_graph(
            WARM_09975_DIR / f"{name}.geff"
        )

        gt_g = load_graph(
            TRAIN_DIR / f"{name}.geff"
        )

        high_keys = node_coordinate_keys(
            high_g
        )

        scale = np.asarray(
            open_dataset(
                TRAIN_DIR / name,
                load_image=False,
            ).scale,
            dtype=float,
        )

        # 1. Frozen G2
        apply_frozen_g2(
            pred_g,
            scale,
        )

        # 2. Confidence-aware pruning
        nodes = (
            pred_g
            .node_attrs(unpack=True)
            .to_pandas()
            .set_index("node_id")
        )

        remove_nodes = set()

        for comp in component_sets(pred_g):

            comp = list(
                map(int, comp)
            )

            if len(comp) > max_comp_size:
                continue

            for node_id in comp:

                row = nodes.loc[node_id]

                key = (
                    int(row["t"]),
                    int(round(row["z"])),
                    int(round(row["y"])),
                    int(round(row["x"])),
                )

                if key not in high_keys:
                    remove_nodes.add(
                        node_id
                    )

        if remove_nodes:
            pred_g.bulk_remove_nodes(
                sorted(remove_nodes)
            )

        # 3. Frozen cleanup
        apply_frozen_component_cleanup(
            pred_g,
            scale,
        )

        # 4. Locked official metric
        er = official_evaluate(
            pred_g,
            gt_g,
            scale=tuple(scale),
            max_distance=7.0,
        )

        recall = node_recall(
            pred_g,
            gt_g,
        )

        m = per_sample_metrics(
            er,
            float(estimated_map[name]),
            recall,
        )

        records.append(m)

        dataset_rows.append({
            "dataset": name,
            "prefix": prefix_map[name],
            "edge_jaccard":
                m["edge_jaccard"],
            "adj_edge_jaccard":
                m["adj_edge_jaccard"],
            "node_recall":
                m["node_recall"],
            "node_ratio":
                m["total_node_ratio"],
        })

    return (
        records,
        pd.DataFrame(dataset_rows),
    )


# ============================================================
# Evaluate K4 and K9
# ============================================================

print("=== 08.14 K4 vs K9 ROBUSTNESS CHECK ===")

variant_records = {}
variant_df = {}

for label, k in [
    ("K4", 4),
    ("K9", 9),
]:
    records, df = evaluate_warm_k(k)

    variant_records[label] = records
    variant_df[label] = df


# ============================================================
# Overall
# ============================================================

overall_rows = []

for label in ["K4", "K9"]:

    records = variant_records[label]
    s = summarise(records)

    overall_rows.append({
        "variant": label,
        "edge_jaccard":
            s["edge_jaccard"],
        "adj_edge_jaccard":
            s["adj_edge_jaccard"],
        "division_jaccard":
            s["division_jaccard"],
        "node_recall":
            s["node_recall"],
        "median_node_ratio":
            float(np.median([
                m["total_node_ratio"]
                for m in records
            ])),
        "score":
            s["score"],
    })

overall_0814 = pd.DataFrame(
    overall_rows
)

print("\nOVERALL")
display(overall_0814)


# ============================================================
# Prefix
# ============================================================

prefix_rows = []

for label in ["K4", "K9"]:

    records = variant_records[label]

    for prefix in [
        "44b6",
        "6bba",
    ]:

        selected = [
            m
            for name, m in zip(
                dataset_names,
                records,
            )
            if prefix_map[name] == prefix
        ]

        s = summarise(
            selected
        )

        ratios = [
            m["total_node_ratio"]
            for name, m in zip(
                dataset_names,
                records,
            )
            if prefix_map[name] == prefix
        ]

        prefix_rows.append({
            "variant": label,
            "prefix": prefix,
            "edge_jaccard":
                s["edge_jaccard"],
            "adj_edge_jaccard":
                s["adj_edge_jaccard"],
            "division_jaccard":
                s["division_jaccard"],
            "node_recall":
                s["node_recall"],
            "median_node_ratio":
                float(np.median(ratios)),
            "score":
                s["score"],
        })

prefix_0814 = (
    pd.DataFrame(prefix_rows)
    .sort_values(
        ["prefix", "variant"]
    )
)

print("\nBY PREFIX")
display(prefix_0814)


# ============================================================
# Paired dataset changes: K9 - K4
# ============================================================

k4 = (
    variant_df["K4"]
    .set_index(
        ["dataset", "prefix"]
    )
)

k9 = (
    variant_df["K9"]
    .set_index(
        ["dataset", "prefix"]
    )
)

paired = (
    k9[
        [
            "edge_jaccard",
            "adj_edge_jaccard",
            "node_recall",
            "node_ratio",
        ]
    ]
    -
    k4[
        [
            "edge_jaccard",
            "adj_edge_jaccard",
            "node_recall",
            "node_ratio",
        ]
    ]
)

paired.columns = [
    "delta_edgeJ",
    "delta_adjEdgeJ",
    "delta_recall",
    "delta_node_ratio",
]

paired = paired.reset_index()

paired_summary = (
    paired
    .groupby("prefix")
    .agg(
        datasets=(
            "dataset",
            "size",
        ),
        mean_delta_edgeJ=(
            "delta_edgeJ",
            "mean",
        ),
        edgeJ_improved_rate=(
            "delta_edgeJ",
            lambda x: float(
                (x > 0).mean()
            ),
        ),
        mean_delta_adjEdgeJ=(
            "delta_adjEdgeJ",
            "mean",
        ),
        adjEdgeJ_improved_rate=(
            "delta_adjEdgeJ",
            lambda x: float(
                (x > 0).mean()
            ),
        ),
        mean_delta_recall=(
            "delta_recall",
            "mean",
        ),
        mean_delta_node_ratio=(
            "delta_node_ratio",
            "mean",
        ),
    )
    .reset_index()
)

print("\nPAIRED K9 - K4")
display(paired_summary)

print()
print(
    "08.14 K4 vs K9 robustness check: COMPLETE"
)

Warm-start checkpoint: ACCEPTED
    score with original K4 = 0.756137

K4:
    robust reference policy

K9:
    Fold0-tuned candidate
    score = 0.757482
    NOT yet promoted to frozen baseline

## 08.15 — Warm-Start det=0.996 Inference

**Status:** COMPLETE

- Detection threshold: `0.996`
- Fold0 datasets: `40/40`
- Runtime: `38.9 min`
- Output:
  `predictions/heqiuyan/official_baseline_fold0_warm1_lr1e5_det0996/split_0`
- High-confidence reference for evaluation: existing `det=0.9975`

In [ ]:
# ============================================================
# RUN — 08.16 Warm-Start Detection Threshold Recalibration
# Purpose:
#   Compare det=0.995 vs det=0.996 under the accepted warm-start
#   checkpoint, using the same det=0.9975 high-confidence reference.
#
#   Candidates:
#     0.995 + K4
#     0.995 + K9
#     0.996 + K0
#     0.996 + K2
#     0.996 + K4
#     0.996 + K9
#
# Changes files: NO
# Runs training: NO
# Runs inference: NO
# Uses GT: YES — Fold0 post-processing calibration
# Keep after running: YES — formal calibration evidence
# ============================================================

# ------------------------------------------------------------
# Locate det=0.996 predictions
# ------------------------------------------------------------

WARM_0996_DIR = find_prediction_dir(
    "official_baseline_fold0_warm1_lr1e5_det0996"
)

assert len(
    list(WARM_0996_DIR.glob("*.geff"))
) == 40


# ------------------------------------------------------------
# Generic evaluator
# ------------------------------------------------------------

def evaluate_threshold_k(
    main_pred_dir,
    high_pred_dir,
    max_comp_size,
):
    records = []
    dataset_rows = []

    nesting_violations = 0

    for name in dataset_names:

        pred_g = load_graph(
            main_pred_dir / f"{name}.geff"
        )

        high_g = load_graph(
            high_pred_dir / f"{name}.geff"
        )

        gt_g = load_graph(
            TRAIN_DIR / f"{name}.geff"
        )

        main_keys = node_coordinate_keys(
            pred_g
        )

        high_keys = node_coordinate_keys(
            high_g
        )

        violations = (
            high_keys - main_keys
        )

        nesting_violations += len(
            violations
        )

        assert not violations, (
            f"Nesting violation in {name}: "
            f"{len(violations)}"
        )

        scale = np.asarray(
            open_dataset(
                TRAIN_DIR / name,
                load_image=False,
            ).scale,
            dtype=float,
        )

        # ----------------------------------------
        # 1. Frozen G2
        # ----------------------------------------

        apply_frozen_g2(
            pred_g,
            scale,
        )

        # ----------------------------------------
        # 2. Confidence-aware pruning
        # ----------------------------------------

        if max_comp_size > 0:

            nodes = (
                pred_g
                .node_attrs(unpack=True)
                .to_pandas()
                .set_index("node_id")
            )

            remove_nodes = set()

            # Components are measured BEFORE pruning.
            for comp in component_sets(
                pred_g
            ):
                comp = list(
                    map(int, comp)
                )

                if len(comp) > max_comp_size:
                    continue

                for node_id in comp:

                    row = nodes.loc[
                        node_id
                    ]

                    key = (
                        int(row["t"]),
                        int(round(row["z"])),
                        int(round(row["y"])),
                        int(round(row["x"])),
                    )

                    if key not in high_keys:
                        remove_nodes.add(
                            node_id
                        )

            if remove_nodes:
                pred_g.bulk_remove_nodes(
                    sorted(remove_nodes)
                )

        # ----------------------------------------
        # 3. Frozen component cleanup
        # ----------------------------------------

        apply_frozen_component_cleanup(
            pred_g,
            scale,
        )

        # ----------------------------------------
        # 4. Locked official metric
        # ----------------------------------------

        er = official_evaluate(
            pred_g,
            gt_g,
            scale=tuple(scale),
            max_distance=7.0,
        )

        recall = node_recall(
            pred_g,
            gt_g,
        )

        m = per_sample_metrics(
            er,
            float(
                estimated_map[name]
            ),
            recall,
        )

        records.append(m)

        dataset_rows.append({
            "dataset":
                name,

            "prefix":
                prefix_map[name],

            "edge_jaccard":
                m["edge_jaccard"],

            "adj_edge_jaccard":
                m["adj_edge_jaccard"],

            "node_recall":
                m["node_recall"],

            "node_ratio":
                m["total_node_ratio"],
        })

    assert nesting_violations == 0

    return (
        records,
        pd.DataFrame(
            dataset_rows
        ),
    )


# ------------------------------------------------------------
# Candidate grid
# ------------------------------------------------------------

CANDIDATES = [
    ("det995_K4", WARM_0995_DIR, 4),
    ("det995_K9", WARM_0995_DIR, 9),

    ("det996_K0", WARM_0996_DIR, 0),
    ("det996_K2", WARM_0996_DIR, 2),
    ("det996_K4", WARM_0996_DIR, 4),
    ("det996_K9", WARM_0996_DIR, 9),
]


# ------------------------------------------------------------
# Evaluate
# ------------------------------------------------------------

print(
    "=== 08.16 WARM DETECTION THRESHOLD "
    "RECALIBRATION ==="
)

variant_records = {}
variant_dfs = {}

overall_rows = []

for label, pred_dir, k in CANDIDATES:

    records, dataset_df = (
        evaluate_threshold_k(
            pred_dir,
            WARM_09975_DIR,
            k,
        )
    )

    variant_records[label] = records
    variant_dfs[label] = dataset_df

    s = summarise(
        records
    )

    overall_rows.append({
        "variant":
            label,

        "edge_jaccard":
            s["edge_jaccard"],

        "adj_edge_jaccard":
            s["adj_edge_jaccard"],

        "division_jaccard":
            s["division_jaccard"],

        "node_recall":
            s["node_recall"],

        "median_node_ratio":
            float(
                np.median([
                    m["total_node_ratio"]
                    for m in records
                ])
            ),

        "score":
            s["score"],
    })


overall_0816 = (
    pd.DataFrame(
        overall_rows
    )
    .sort_values(
        "score",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# Prefix summaries
# ------------------------------------------------------------

prefix_rows = []

for label, _, _ in CANDIDATES:

    records = (
        variant_records[label]
    )

    for prefix in [
        "44b6",
        "6bba",
    ]:

        selected = [
            m
            for name, m in zip(
                dataset_names,
                records,
            )
            if prefix_map[name]
            == prefix
        ]

        s = summarise(
            selected
        )

        ratios = [
            m["total_node_ratio"]
            for name, m in zip(
                dataset_names,
                records,
            )
            if prefix_map[name]
            == prefix
        ]

        prefix_rows.append({
            "variant":
                label,

            "prefix":
                prefix,

            "edge_jaccard":
                s["edge_jaccard"],

            "adj_edge_jaccard":
                s["adj_edge_jaccard"],

            "division_jaccard":
                s["division_jaccard"],

            "node_recall":
                s["node_recall"],

            "median_node_ratio":
                float(
                    np.median(ratios)
                ),

            "score":
                s["score"],
        })


prefix_0816 = (
    pd.DataFrame(
        prefix_rows
    )
    .sort_values(
        [
            "prefix",
            "score",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

print()
print("OVERALL")
display(
    overall_0816
)

print()
print("BY PREFIX")
display(
    prefix_0816
)

best = overall_0816.iloc[0]

print()
print(
    "Best candidate:",
    best["variant"],
)

print(
    "Best score:",
    f"{best['score']:.6f}",
)

print(
    "Delta vs warm det=.995 K4:",
    f"{best['score'] - 0.756137:+.6f}",
)

print(
    "Delta vs warm det=.995 K9:",
    f"{best['score'] - 0.757482:+.6f}",
)

print()
print(
    "08.16 detection threshold "
    "recalibration: COMPLETE"
)

## 08.16 — Detection Threshold Recalibration

**Status: COMPLETE**

Warm-start checkpoint detection-threshold comparison confirms:

- `det=0.995 + K9`: **0.757482**
- `det=0.996 + K9`: `0.757129`

Although `0.996` reduces median node ratio slightly, it lowers:
- raw Edge Jaccard
- adjusted Edge Jaccard
- division Jaccard
- node recall
- overall score

Therefore:

**Frozen development baseline-v5**
- checkpoint: `warm1_lr1e5`
- detection threshold: `0.995`
- high-confidence threshold: `0.9975`
- confidence component cutoff: `K=9`
- G2 geometry: unchanged
- singleton cleanup: enabled
- 2-node maximum displacement: `3.5 µm`
- Fold0 score: **0.757482**

Improvement vs baseline-v4:
`+0.013230`

In [ ]:
# ============================================================
# RUN — 08.17 Warm-Start G2 p2 Recalibration
# Purpose:
#   Recalibrate only the G2 second-child edge-probability
#   threshold for the accepted warm-start detector.
#
# Frozen:
#   det = 0.995
#   high-confidence threshold = 0.9975
#   confidence pruning K = 9
#   all G2 geometry thresholds unchanged
#   singleton cleanup unchanged
#   2-node max displacement = 3.5 um
#
# Changes files: NO
# Runs training: NO
# Runs inference: NO
# Uses GT: YES — Fold0 post-processing calibration
# Keep after running: YES — formal calibration evidence
# ============================================================

# ------------------------------------------------------------
# Preflight: reuse already validated helpers
# ------------------------------------------------------------

REQUIRED = [
    "apply_frozen_g2",
    "apply_frozen_component_cleanup",
    "component_sets",
    "node_coordinate_keys",
    "load_graph",
    "official_evaluate",
    "node_recall",
    "per_sample_metrics",
    "summarise",
    "open_dataset",
    "WARM_0995_DIR",
    "WARM_09975_DIR",
    "TRAIN_DIR",
    "dataset_names",
    "estimated_map",
    "prefix_map",
]

missing = [
    name
    for name in REQUIRED
    if name not in globals()
]

assert not missing, (
    "Required frozen helpers are missing from the current kernel:\n"
    + "\n".join(missing)
)

# Accepted baseline-v5 settings
CONF_COMPONENT_LIMIT_0817 = 9

# Keep the current frozen value so it can be restored.
ORIGINAL_G2_P2_MIN = float(
    G2_P2_MIN
)

P2_CANDIDATES = [
    0.35,
    0.45,
    0.55,  # current frozen value
    0.65,
    0.75,
    0.85,
]


# ------------------------------------------------------------
# Evaluate one p2 value
# ------------------------------------------------------------

def evaluate_p2_candidate(p2_min):

    global G2_P2_MIN
    G2_P2_MIN = float(p2_min)

    records = []
    dataset_rows = []

    total_forks_after_g2 = 0

    for name in dataset_names:

        pred_g = load_graph(
            WARM_0995_DIR / f"{name}.geff"
        )

        high_g = load_graph(
            WARM_09975_DIR / f"{name}.geff"
        )

        gt_g = load_graph(
            TRAIN_DIR / f"{name}.geff"
        )

        high_keys = node_coordinate_keys(
            high_g
        )

        scale = np.asarray(
            open_dataset(
                TRAIN_DIR / name,
                load_image=False,
            ).scale,
            dtype=float,
        )

        # ----------------------------------------
        # 1. G2 — only p2 threshold varies
        # ----------------------------------------

        apply_frozen_g2(
            pred_g,
            scale,
        )

        # Count surviving predicted forks after G2.
        edges_after_g2 = (
            pred_g
            .edge_attrs(
                attr_keys=["edge_prob"],
                unpack=True,
            )
            .to_pandas()
        )

        if len(edges_after_g2):
            outdegree = (
                edges_after_g2
                .groupby("source_id")
                .size()
            )

            total_forks_after_g2 += int(
                (outdegree == 2).sum()
            )

        # ----------------------------------------
        # 2. Frozen K=9 confidence pruning
        # ----------------------------------------

        nodes = (
            pred_g
            .node_attrs(unpack=True)
            .to_pandas()
            .set_index("node_id")
        )

        remove_nodes = set()

        for comp in component_sets(
            pred_g
        ):
            comp = list(
                map(int, comp)
            )

            if len(comp) > CONF_COMPONENT_LIMIT_0817:
                continue

            for node_id in comp:

                row = nodes.loc[
                    node_id
                ]

                key = (
                    int(row["t"]),
                    int(round(row["z"])),
                    int(round(row["y"])),
                    int(round(row["x"])),
                )

                if key not in high_keys:
                    remove_nodes.add(
                        node_id
                    )

        if remove_nodes:
            pred_g.bulk_remove_nodes(
                sorted(remove_nodes)
            )

        # ----------------------------------------
        # 3. Frozen component cleanup
        # ----------------------------------------

        apply_frozen_component_cleanup(
            pred_g,
            scale,
        )

        # ----------------------------------------
        # 4. Locked official metric
        # ----------------------------------------

        er = official_evaluate(
            pred_g,
            gt_g,
            scale=tuple(scale),
            max_distance=7.0,
        )

        recall = node_recall(
            pred_g,
            gt_g,
        )

        m = per_sample_metrics(
            er,
            float(
                estimated_map[name]
            ),
            recall,
        )

        records.append(m)

        dataset_rows.append({
            "dataset":
                name,

            "prefix":
                prefix_map[name],

            "edge_jaccard":
                m["edge_jaccard"],

            "adj_edge_jaccard":
                m["adj_edge_jaccard"],

            "node_recall":
                m["node_recall"],

            "node_ratio":
                m["total_node_ratio"],
        })

    s = summarise(
        records
    )

    return {
        "p2_min":
            float(p2_min),

        "edge_jaccard":
            s["edge_jaccard"],

        "adj_edge_jaccard":
            s["adj_edge_jaccard"],

        "division_jaccard":
            s["division_jaccard"],

        "node_recall":
            s["node_recall"],

        "median_node_ratio":
            float(
                np.median([
                    m["total_node_ratio"]
                    for m in records
                ])
            ),

        "predicted_forks_after_g2":
            total_forks_after_g2,

        "score":
            s["score"],

        "_records":
            records,

        "_dataset_df":
            pd.DataFrame(
                dataset_rows
            ),
    }


# ------------------------------------------------------------
# Sweep
# ------------------------------------------------------------

print("=== 08.17 WARM-START G2 p2 RECALIBRATION ===")
print()

results_0817 = []

try:
    for p2 in P2_CANDIDATES:
        result = evaluate_p2_candidate(
            p2
        )

        results_0817.append(
            result
        )

finally:
    # Always restore the frozen global setting.
    G2_P2_MIN = ORIGINAL_G2_P2_MIN


# ------------------------------------------------------------
# Overall table
# ------------------------------------------------------------

overall_0817 = pd.DataFrame([
    {
        key: value
        for key, value in r.items()
        if not key.startswith("_")
    }
    for r in results_0817
])

overall_0817 = (
    overall_0817
    .sort_values(
        "score",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)

print("OVERALL")
display(
    overall_0817
)


# ------------------------------------------------------------
# Prefix table
# ------------------------------------------------------------

prefix_rows = []

for result in results_0817:

    p2 = result["p2_min"]
    records = result["_records"]

    for prefix in [
        "44b6",
        "6bba",
    ]:

        selected = [
            m
            for name, m in zip(
                dataset_names,
                records,
            )
            if prefix_map[name]
            == prefix
        ]

        s = summarise(
            selected
        )

        ratios = [
            m["total_node_ratio"]
            for name, m in zip(
                dataset_names,
                records,
            )
            if prefix_map[name]
            == prefix
        ]

        prefix_rows.append({
            "p2_min":
                p2,

            "prefix":
                prefix,

            "edge_jaccard":
                s["edge_jaccard"],

            "adj_edge_jaccard":
                s["adj_edge_jaccard"],

            "division_jaccard":
                s["division_jaccard"],

            "node_recall":
                s["node_recall"],

            "median_node_ratio":
                float(
                    np.median(ratios)
                ),

            "score":
                s["score"],
        })


prefix_0817 = (
    pd.DataFrame(
        prefix_rows
    )
    .sort_values(
        [
            "prefix",
            "score",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)

print()
print("BY PREFIX")
display(
    prefix_0817
)


# ------------------------------------------------------------
# Best
# ------------------------------------------------------------

best = overall_0817.iloc[0]

print()
print(
    "Best p2_min:",
    best["p2_min"],
)

print(
    "Best score:",
    f"{best['score']:.6f}",
)

print(
    "Delta vs baseline-v5 p2=0.55:",
    f"{best['score'] - 0.757482:+.6f}",
)

print()
print(
    "G2_P2_MIN restored to:",
    G2_P2_MIN,
)

print()
print(
    "08.17 G2 p2 recalibration: COMPLETE"
)

In [ ]:
# ============================================================
# RESULT — 08.18 G2 p2=.55 vs .75 Robustness Check
# Purpose:
#   Compare the frozen p2=.55 policy against candidate p2=.75
#   using the already-computed 08.17 per-dataset results.
#
# Changes files: NO
# Runs training: NO
# Runs inference: NO
# Uses GT: YES — reuses existing Fold0 evaluation results
# Keep after running: YES
# ============================================================

# ------------------------------------------------------------
# Recover existing 08.17 results
# ------------------------------------------------------------

assert "results_0817" in globals(), (
    "08.17 results are not present in the current kernel."
)

result_by_p2 = {
    float(r["p2_min"]): r
    for r in results_0817
}

assert 0.55 in result_by_p2
assert 0.75 in result_by_p2

r55 = result_by_p2[0.55]
r75 = result_by_p2[0.75]

df55 = (
    r55["_dataset_df"]
    .set_index(["dataset", "prefix"])
)

df75 = (
    r75["_dataset_df"]
    .set_index(["dataset", "prefix"])
)


# ------------------------------------------------------------
# Paired dataset deltas: .75 - .55
# ------------------------------------------------------------

paired_0818 = pd.DataFrame(
    index=df55.index
)

for metric in [
    "edge_jaccard",
    "adj_edge_jaccard",
    "node_recall",
    "node_ratio",
]:
    paired_0818[
        f"delta_{metric}"
    ] = (
        df75[metric]
        - df55[metric]
    )

paired_0818 = (
    paired_0818
    .reset_index()
)


# ------------------------------------------------------------
# Overall paired robustness
# ------------------------------------------------------------

overall_paired_0818 = pd.DataFrame([{
    "datasets":
        len(paired_0818),

    "edgeJ_improved_rate":
        float(
            (
                paired_0818[
                    "delta_edge_jaccard"
                ] > 0
            ).mean()
        ),

    "adjEdgeJ_improved_rate":
        float(
            (
                paired_0818[
                    "delta_adj_edge_jaccard"
                ] > 0
            ).mean()
        ),

    "mean_delta_edgeJ":
        float(
            paired_0818[
                "delta_edge_jaccard"
            ].mean()
        ),

    "median_delta_edgeJ":
        float(
            paired_0818[
                "delta_edge_jaccard"
            ].median()
        ),

    "mean_delta_adjEdgeJ":
        float(
            paired_0818[
                "delta_adj_edge_jaccard"
            ].mean()
        ),

    "median_delta_adjEdgeJ":
        float(
            paired_0818[
                "delta_adj_edge_jaccard"
            ].median()
        ),

    "mean_delta_recall":
        float(
            paired_0818[
                "delta_node_recall"
            ].mean()
        ),

    "mean_delta_node_ratio":
        float(
            paired_0818[
                "delta_node_ratio"
            ].mean()
        ),
}])


# ------------------------------------------------------------
# Prefix-level paired robustness
# ------------------------------------------------------------

prefix_paired_0818 = (
    paired_0818
    .groupby("prefix")
    .agg(
        datasets=(
            "dataset",
            "size",
        ),

        edgeJ_improved_rate=(
            "delta_edge_jaccard",
            lambda x: float(
                (x > 0).mean()
            ),
        ),

        adjEdgeJ_improved_rate=(
            "delta_adj_edge_jaccard",
            lambda x: float(
                (x > 0).mean()
            ),
        ),

        mean_delta_edgeJ=(
            "delta_edge_jaccard",
            "mean",
        ),

        median_delta_edgeJ=(
            "delta_edge_jaccard",
            "median",
        ),

        mean_delta_adjEdgeJ=(
            "delta_adj_edge_jaccard",
            "mean",
        ),

        median_delta_adjEdgeJ=(
            "delta_adj_edge_jaccard",
            "median",
        ),

        mean_delta_recall=(
            "delta_node_recall",
            "mean",
        ),

        mean_delta_node_ratio=(
            "delta_node_ratio",
            "mean",
        ),
    )
    .reset_index()
)


# ------------------------------------------------------------
# Fork suppression summary
# ------------------------------------------------------------

fork_summary_0818 = pd.DataFrame([
    {
        "p2_min":
            0.55,

        "predicted_forks_after_g2":
            r55[
                "predicted_forks_after_g2"
            ],

        "division_jaccard":
            r55[
                "division_jaccard"
            ],

        "score":
            r55[
                "score"
            ],
    },
    {
        "p2_min":
            0.75,

        "predicted_forks_after_g2":
            r75[
                "predicted_forks_after_g2"
            ],

        "division_jaccard":
            r75[
                "division_jaccard"
            ],

        "score":
            r75[
                "score"
            ],
    },
])


# ------------------------------------------------------------
# Output
# ------------------------------------------------------------

print(
    "=== 08.18 p2=.55 vs .75 ROBUSTNESS ==="
)

print()
print("OVERALL PAIRED")
display(
    overall_paired_0818
)

print()
print("BY PREFIX")
display(
    prefix_paired_0818
)

print()
print("FORK / DIVISION TRADEOFF")
display(
    fork_summary_0818
)

print()
print(
    "08.18 p2 robustness check: COMPLETE"
)

In [ ]:
# ============================================================
# RUN — 08.19 Fine G2 p2 Local Sweep
# Purpose:
#   Resolve the local optimum around coarse-sweep winner p2=.75
#   while keeping the accepted warm-start baseline fixed.
#
# Frozen:
#   checkpoint = warm1_lr1e5
#   det = 0.995
#   high-confidence = 0.9975
#   confidence pruning K = 9
#   all G2 geometry thresholds unchanged
#   singleton cleanup unchanged
#   2-node max displacement = 3.5 um
#
# Changes files: NO
# Runs training: NO
# Runs inference: NO
# Uses GT: YES — Fold0 post-processing calibration
# Keep after running: YES — formal calibration evidence
# ============================================================

assert "evaluate_p2_candidate" in globals(), (
    "08.17 evaluate_p2_candidate() is missing "
    "from the current kernel."
)

ORIGINAL_G2_P2_MIN_0819 = float(
    G2_P2_MIN
)

P2_FINE = [
    0.700,
    0.725,
    0.750,
    0.775,
    0.800,
]

results_0819 = []

print("=== 08.19 FINE G2 p2 SWEEP ===")
print()

try:
    for p2 in P2_FINE:
        result = evaluate_p2_candidate(
            p2
        )
        results_0819.append(
            result
        )

finally:
    G2_P2_MIN = ORIGINAL_G2_P2_MIN_0819


# ============================================================
# Overall
# ============================================================

overall_0819 = pd.DataFrame([
    {
        key: value
        for key, value in r.items()
        if not key.startswith("_")
    }
    for r in results_0819
])

overall_0819 = (
    overall_0819
    .sort_values(
        "score",
        ascending=False,
    )
    .reset_index(drop=True)
)

print("OVERALL")
display(
    overall_0819[
        [
            "p2_min",
            "edge_jaccard",
            "adj_edge_jaccard",
            "division_jaccard",
            "node_recall",
            "median_node_ratio",
            "predicted_forks_after_g2",
            "score",
        ]
    ]
)


# ============================================================
# Prefix
# ============================================================

prefix_rows = []

for result in results_0819:

    p2 = result["p2_min"]
    records = result["_records"]

    for prefix in [
        "44b6",
        "6bba",
    ]:

        selected = [
            m
            for name, m in zip(
                dataset_names,
                records,
            )
            if prefix_map[name] == prefix
        ]

        s = summarise(
            selected
        )

        ratios = [
            m["total_node_ratio"]
            for name, m in zip(
                dataset_names,
                records,
            )
            if prefix_map[name] == prefix
        ]

        prefix_rows.append({
            "p2_min":
                p2,

            "prefix":
                prefix,

            "edge_jaccard":
                s["edge_jaccard"],

            "adj_edge_jaccard":
                s["adj_edge_jaccard"],

            "division_jaccard":
                s["division_jaccard"],

            "node_recall":
                s["node_recall"],

            "median_node_ratio":
                float(
                    np.median(ratios)
                ),

            "score":
                s["score"],
        })


prefix_0819 = (
    pd.DataFrame(prefix_rows)
    .sort_values(
        [
            "prefix",
            "p2_min",
        ]
    )
    .reset_index(drop=True)
)

print()
print("BY PREFIX")
display(
    prefix_0819
)


# ============================================================
# Local stability
# ============================================================

best = overall_0819.iloc[0]

print()
print("Best local p2:", best["p2_min"])
print(
    "Best local score:",
    f"{best['score']:.6f}",
)
print(
    "Delta vs p2=.75 coarse result:",
    f"{best['score'] - 0.759197:+.6f}",
)
print(
    "Surviving forks:",
    int(best["predicted_forks_after_g2"]),
)

print()
print(
    "G2_P2_MIN restored to:",
    G2_P2_MIN,
)

print()
print(
    "08.19 fine p2 sweep: COMPLETE"
)

## 08.20 — Final G2 p2 Calibration

**Status: COMPLETE — p2 tuning closed**

Final checkpoint-specific G2 threshold:

`p2_min = 0.80`

Fold0 score: **0.760314**

At `p2 >= 0.81`, Division Jaccard collapses from `0.033898`
to `0`, while Edge Jaccard remains nearly unchanged.

Therefore `0.80` is retained as the final G2 operating point for
the current warm-start checkpoint.

No further p2 tuning.

## 08.21 — Warm-Start Stage 2 Training

**Status: COMPLETE**

Initialization:
`official_baseline_fold0_warm1_lr1e5`

Warm1 SHA256:
`018ec5d91104a880b61351e5bbe87ef9e2fb102bc69d14df17b6ae95e6ca0b00`

Training:
- additional full Fold0 epochs: `1`
- optimizer: fresh AdamW
- learning rate: `1e-5`
- batch size: `8`
- seed: `1640`

Validation:
- edge loss: `0.0005`
- detection loss: `0.0111`
- accuracy: `0.9997`
- recall: `0.9139`
- internal acc × recall: `0.9136`

Runtime: `15.28 h`

Output:
`models/official_baseline/official_baseline_fold0_warm2_lr1e5/split_0/edge_predictor_best.pth`

Warm2 SHA256:
`31d209bfde1fd5bbf3b2a661cdb4a25bb72922b832d315ee5a75cc32f44656c1`

Warm1 checkpoint remained unchanged.

Next:
fixed-policy Fold0 comparison against warm1 baseline-v6.

In [ ]:
# ============================================================
# RESULT — 08.22 Warm2 Checkpoint / Training Audit
# Purpose:
#   Verify the completed warm2 checkpoint, preserve its SHA256,
#   and extract the final training/validation summary from the
#   completed 08.21 training log.
#
# Changes files: NO
# Runs training: NO
# Runs inference: NO
# Uses GT: NO
# Keep after running: YES
# ============================================================

from pathlib import Path
import hashlib

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

WARM1 = (
    PROJECT
    / "models/official_baseline/"
      "official_baseline_fold0_warm1_lr1e5/"
      "split_0/edge_predictor_best.pth"
)

WARM2 = (
    PROJECT
    / "models/official_baseline/"
      "official_baseline_fold0_warm2_lr1e5/"
      "split_0/edge_predictor_best.pth"
)

LOG_PATH = (
    PROJECT
    / "reports/training_logs/"
      "08_21_warm2_lr1e5.log"
)


def sha256(path: Path) -> str:
    h = hashlib.sha256()

    with path.open("rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):
            h.update(chunk)

    return h.hexdigest()


# ------------------------------------------------------------
# Preflight
# ------------------------------------------------------------

assert WARM1.exists(), WARM1
assert WARM2.exists(), WARM2
assert LOG_PATH.exists(), LOG_PATH

EXPECTED_WARM1_SHA = (
    "018ec5d91104a880b61351e5bbe87ef9"
    "e2fb102bc69d14df17b6ae95e6ca0b00"
)

warm1_sha = sha256(WARM1)
warm2_sha = sha256(WARM2)

assert warm1_sha == EXPECTED_WARM1_SHA, (
    "Warm1 checkpoint changed unexpectedly."
)

assert warm2_sha != warm1_sha, (
    "Warm2 checkpoint is byte-identical to warm1."
)


# ------------------------------------------------------------
# Parse concise training summary
# ------------------------------------------------------------

lines = LOG_PATH.read_text(
    errors="replace"
).splitlines()

epoch_lines = [
    line.strip()
    for line in lines
    if (
        "Epoch " in line
        and (
            "edge=" in line
            or "recall=" in line
            or "best=" in line
        )
    )
]

warning_lines = [
    line.strip()
    for line in lines
    if (
        "Traceback" in line
        or "RuntimeError" in line
        or "ERROR" in line
    )
]


# ------------------------------------------------------------
# Result
# ------------------------------------------------------------

print("=== 08.22 WARM2 TRAINING AUDIT ===")
print()

print("Warm1 SHA256:")
print(warm1_sha)

print()
print("Warm2 SHA256:")
print(warm2_sha)

print()
print(
    "Warm2 size:",
    f"{WARM2.stat().st_size / 1024 / 1024:.2f} MB",
)

print()
print("Final epoch summary:")

if epoch_lines:
    for line in epoch_lines[-3:]:
        print(line)
else:
    print("No concise Epoch summary line detected.")

print()
print(
    "Fatal-error markers:",
    len(warning_lines),
)

if warning_lines:
    for line in warning_lines[-5:]:
        print(line)

print()
print("Warm1 unchanged: YES")
print("Warm2 checkpoint exists: YES")
print("08.22 warm2 checkpoint audit: COMPLETE")

In [ ]:
# ============================================================
# RUN — 08.24 Warm1 vs Warm2 Fixed-Policy Evaluation
# Purpose:
#   Compare warm1 and warm2 under EXACTLY the same current
#   baseline-v6 post-processing policy:
#
#       det = 0.995
#       high-confidence = 0.9975
#       confidence pruning K = 9
#       G2 p2_min = 0.80
#       all other G2 geometry frozen
#       singleton cleanup
#       remove 2-node components > 3.5 um
#
#   Frozen helper implementations are extracted directly from
#   the validated Stage-07 notebook to avoid semantic drift.
#
# Changes files: NO
# Runs training: NO
# Runs inference: NO
# Uses GT: YES — Fold0 evaluation only
# Keep after running: YES — formal model comparison evidence
# ============================================================

from pathlib import Path
from collections import defaultdict
import ast
import json
import sys
import warnings

import numpy as np
import pandas as pd
import tracksdata as td


# ============================================================
# Paths
# ============================================================

PROJECT = Path(
    "/Users/heqiuyan/Desktop/biohub-cell-tracking"
)

TRAIN_DIR = (
    PROJECT
    / "data/raw/competition/"
      "biohub-cell-tracking-during-development/train"
)

CV_PATH = (
    PROJECT
    / "configs/cv_manifest.csv"
)

NOTEBOOK_07 = (
    PROJECT
    / "notebooks/07_detection_threshold_sweep.ipynb"
)

OFFICIAL_SRC = (
    PROJECT
    / "external/official/src"
)

OFFICIAL_SCRIPTS = (
    PROJECT
    / "external/official/scripts"
)

for p in [
    OFFICIAL_SRC,
    OFFICIAL_SCRIPTS,
]:
    if str(p) not in sys.path:
        sys.path.insert(
            0,
            str(p),
        )


# ============================================================
# Locked official metric
# ============================================================

from tracking_cellmot.io import open_dataset
from tracking_cellmot.metrics import (
    evaluate as official_evaluate,
    node_recall,
    per_sample_metrics,
    summarise,
)

warnings.filterwarnings(
    "ignore",
    message="Graph already matched, overwriting previous matching.",
)


# ============================================================
# Load Stage-07 notebook
# ============================================================

assert NOTEBOOK_07.exists(), NOTEBOOK_07

with NOTEBOOK_07.open(
    "r",
    encoding="utf-8",
) as f:
    nb07 = json.load(f)


def import_functions_from_cell(
    cell_index,
    function_names,
):
    """
    Extract only named function definitions from the original
    validated Stage-07 notebook and execute those definitions
    in the current notebook namespace.
    """

    source = "".join(
        nb07["cells"][cell_index]["source"]
    )

    tree = ast.parse(source)

    selected = [
        node
        for node in tree.body
        if (
            isinstance(
                node,
                (
                    ast.FunctionDef,
                    ast.AsyncFunctionDef,
                ),
            )
            and node.name in function_names
        )
    ]

    found = {
        node.name
        for node in selected
    }

    missing = (
        set(function_names)
        - found
    )

    assert not missing, (
        f"Missing functions in Cell {cell_index}: "
        f"{missing}"
    )

    module = ast.Module(
        body=selected,
        type_ignores=[],
    )

    ast.fix_missing_locations(
        module
    )

    exec(
        compile(
            module,
            filename=(
                f"{NOTEBOOK_07.name}"
                f":cell_{cell_index}"
            ),
            mode="exec",
        ),
        globals(),
    )


# ------------------------------------------------------------
# Exact frozen helper sources
# ------------------------------------------------------------

import_functions_from_cell(
    0,
    [
        "load_graph",
        "apply_frozen_g2",
    ],
)

import_functions_from_cell(
    3,
    [
        "component_sets",
    ],
)

import_functions_from_cell(
    16,
    [
        "node_coordinate_keys",
    ],
)

import_functions_from_cell(
    19,
    [
        "apply_frozen_component_cleanup",
        "apply_low_conf_component_pruning",
    ],
)


# ============================================================
# Current baseline-v6 policy
# ============================================================

# Only p2 changed from the original Stage-07 G2 policy.
G2_P2_MIN = 0.80

G2_DAUGHTER_SEP_MIN_UM = 6.0
G2_ANGLE_MIN_DEG = 120.0
G2_MAX_PARENT_DAUGHTER_UM = 8.0
G2_DISTANCE_BALANCE_MAX = 0.50

TWO_NODE_MAX_DIST_UM = 3.5

CONF_COMPONENT_LIMIT = 9

WARM1_FROZEN_SCORE = 0.760314


# ============================================================
# Fold0 metadata
# ============================================================

cv_meta = pd.read_csv(
    CV_PATH
)

fold0 = (
    cv_meta[
        cv_meta["fold"] == 0
    ]
    .copy()
)

assert len(fold0) == 40

dataset_names = sorted(
    fold0["dataset"]
)

prefix_map = dict(
    zip(
        fold0["dataset"],
        fold0["prefix"],
    )
)

estimated_map = dict(
    zip(
        fold0["dataset"],
        fold0["estimated_nodes"],
    )
)


# ============================================================
# Locate prediction directories
# ============================================================

PRED_ROOT = (
    PROJECT
    / "predictions"
)


def find_prediction_dir(
    method,
):
    candidates = []

    for split_dir in (
        PRED_ROOT.rglob("split_0")
    ):
        if (
            split_dir.parent.name
            != method
        ):
            continue

        geffs = list(
            split_dir.glob("*.geff")
        )

        if len(geffs) == 40:
            candidates.append(
                split_dir
            )

    assert len(candidates) == 1, (
        f"Expected one complete directory for {method}.\n"
        f"Found: {candidates}"
    )

    return candidates[0]


WARM1_0995 = find_prediction_dir(
    "official_baseline_fold0_warm1_lr1e5_det0995"
)

WARM1_09975 = find_prediction_dir(
    "official_baseline_fold0_warm1_lr1e5_det09975"
)

WARM2_0995 = find_prediction_dir(
    "official_baseline_fold0_warm2_lr1e5_det0995"
)

WARM2_09975 = find_prediction_dir(
    "official_baseline_fold0_warm2_lr1e5_det09975"
)


# ============================================================
# Fixed-policy evaluator
# ============================================================

def evaluate_fixed_policy(
    pred_0995_dir,
    pred_09975_dir,
):

    records = []
    dataset_rows = []

    total_conf_removed = 0
    total_cleanup_removed = 0

    for name in dataset_names:

        pred_g = load_graph(
            pred_0995_dir
            / f"{name}.geff"
        )

        high_g = load_graph(
            pred_09975_dir
            / f"{name}.geff"
        )

        gt_g = load_graph(
            TRAIN_DIR
            / f"{name}.geff"
        )

        main_keys = (
            node_coordinate_keys(
                pred_g
            )
        )

        high_keys = (
            node_coordinate_keys(
                high_g
            )
        )

        violations = (
            high_keys
            - main_keys
        )

        assert not violations, (
            f"Threshold nesting violation "
            f"in {name}: {len(violations)}"
        )

        scale = np.asarray(
            open_dataset(
                TRAIN_DIR / name,
                load_image=False,
            ).scale,
            dtype=float,
        )

        # ----------------------------------------
        # 1. Exact frozen G2, current p2=.80
        # ----------------------------------------

        apply_frozen_g2(
            pred_g,
            scale,
        )

        # ----------------------------------------
        # 2. Exact confidence-aware K=9 pruning
        # ----------------------------------------

        conf_stats = (
            apply_low_conf_component_pruning(
                pred_g,
                high_keys,
                CONF_COMPONENT_LIMIT,
            )
        )

        total_conf_removed += (
            conf_stats[
                "low_conf_nodes_removed"
            ]
        )

        # ----------------------------------------
        # 3. Exact frozen component cleanup
        # ----------------------------------------

        cleanup_stats = (
            apply_frozen_component_cleanup(
                pred_g,
                scale,
            )
        )

        total_cleanup_removed += (
            cleanup_stats[
                "nodes_removed"
            ]
        )

        # ----------------------------------------
        # 4. Locked official metric
        # ----------------------------------------

        er = official_evaluate(
            pred_g,
            gt_g,
            scale=tuple(scale),
            max_distance=7.0,
        )

        recall = node_recall(
            pred_g,
            gt_g,
        )

        m = per_sample_metrics(
            er,
            float(
                estimated_map[name]
            ),
            recall,
        )

        records.append(
            m
        )

        # IMPORTANT:
        # division_jaccard is NOT a per-sample m field.
        dataset_rows.append({
            "dataset":
                name,

            "prefix":
                prefix_map[name],

            "edge_jaccard":
                m["edge_jaccard"],

            "adj_edge_jaccard":
                m["adj_edge_jaccard"],

            "node_recall":
                m["node_recall"],

            "node_ratio":
                m["total_node_ratio"],
        })

    s = summarise(
        records
    )

    overall = {
        "edge_jaccard":
            s["edge_jaccard"],

        "adj_edge_jaccard":
            s["adj_edge_jaccard"],

        "division_jaccard":
            s["division_jaccard"],

        "node_recall":
            s["node_recall"],

        "median_node_ratio":
            float(
                np.median([
                    m["total_node_ratio"]
                    for m in records
                ])
            ),

        "score":
            s["score"],

        "conf_nodes_removed":
            total_conf_removed,

        "cleanup_nodes_removed":
            total_cleanup_removed,
    }

    dataset_df = pd.DataFrame(
        dataset_rows
    )

    prefix_rows = []

    for prefix in [
        "44b6",
        "6bba",
    ]:

        selected = [
            m
            for name, m in zip(
                dataset_names,
                records,
            )
            if prefix_map[name]
            == prefix
        ]

        ps = summarise(
            selected
        )

        ratios = [
            m["total_node_ratio"]
            for name, m in zip(
                dataset_names,
                records,
            )
            if prefix_map[name]
            == prefix
        ]

        prefix_rows.append({
            "prefix":
                prefix,

            "edge_jaccard":
                ps["edge_jaccard"],

            "adj_edge_jaccard":
                ps["adj_edge_jaccard"],

            "division_jaccard":
                ps["division_jaccard"],

            "node_recall":
                ps["node_recall"],

            "median_node_ratio":
                float(
                    np.median(ratios)
                ),

            "score":
                ps["score"],
        })

    return (
        overall,
        pd.DataFrame(prefix_rows),
        dataset_df,
    )


# ============================================================
# Warm1 reproduction gate
# ============================================================

print(
    "=== 08.24 WARM1 vs WARM2 "
    "FIXED-POLICY EVALUATION ==="
)

print()
print(
    "Reproducing warm1 baseline-v6..."
)

warm1_overall, warm1_prefix, warm1_df = (
    evaluate_fixed_policy(
        WARM1_0995,
        WARM1_09975,
    )
)

warm1_score = float(
    warm1_overall["score"]
)

reproduction_delta = (
    warm1_score
    - WARM1_FROZEN_SCORE
)

print(
    "Warm1 reproduced:",
    f"{warm1_score:.6f}",
)

print(
    "Difference vs 0.760314:",
    f"{reproduction_delta:+.6f}",
)

assert abs(
    reproduction_delta
) < 5e-6, (
    "Warm1 baseline-v6 reproduction failed. "
    "Stop before evaluating warm2."
)


# ============================================================
# Warm2 — SAME policy, zero recalibration
# ============================================================

print()
print(
    "Reproduction gate: OK"
)

print(
    "Evaluating warm2 with identical policy..."
)

warm2_overall, warm2_prefix, warm2_df = (
    evaluate_fixed_policy(
        WARM2_0995,
        WARM2_09975,
    )
)


# ============================================================
# Overall comparison
# ============================================================

comparison_0824 = pd.DataFrame([
    {
        "model":
            "warm1",

        **warm1_overall,
    },
    {
        "model":
            "warm2",

        **warm2_overall,
    },
])

warm2_score = float(
    warm2_overall["score"]
)

score_delta = (
    warm2_score
    - warm1_score
)


# ============================================================
# Paired per-dataset changes
# ============================================================

a = (
    warm1_df
    .set_index(
        [
            "dataset",
            "prefix",
        ]
    )
)

b = (
    warm2_df
    .set_index(
        [
            "dataset",
            "prefix",
        ]
    )
)

paired = pd.DataFrame(
    index=a.index
)

for metric in [
    "edge_jaccard",
    "adj_edge_jaccard",
    "node_recall",
    "node_ratio",
]:
    paired[
        f"delta_{metric}"
    ] = (
        b[metric]
        - a[metric]
    )

paired = (
    paired
    .reset_index()
)

paired_summary = (
    paired
    .groupby("prefix")
    .agg(
        datasets=(
            "dataset",
            "size",
        ),

        edgeJ_improved_rate=(
            "delta_edge_jaccard",
            lambda x: float(
                (x > 0).mean()
            ),
        ),

        adjEdgeJ_improved_rate=(
            "delta_adj_edge_jaccard",
            lambda x: float(
                (x > 0).mean()
            ),
        ),

        mean_delta_edgeJ=(
            "delta_edge_jaccard",
            "mean",
        ),

        mean_delta_adjEdgeJ=(
            "delta_adj_edge_jaccard",
            "mean",
        ),

        mean_delta_recall=(
            "delta_node_recall",
            "mean",
        ),

        mean_delta_node_ratio=(
            "delta_node_ratio",
            "mean",
        ),
    )
    .reset_index()
)


# ============================================================
# Results
# ============================================================

print()
print("=" * 72)
print("08.24 RESULT")
print("=" * 72)

print()
print("OVERALL")

display(
    comparison_0824[
        [
            "model",
            "edge_jaccard",
            "adj_edge_jaccard",
            "division_jaccard",
            "node_recall",
            "median_node_ratio",
            "score",
        ]
    ]
)

print(
    "Warm2 - Warm1 score delta:",
    f"{score_delta:+.6f}",
)

print()
print("WARM2 BY PREFIX")

display(
    warm2_prefix
)

print()
print("PAIRED WARM2 - WARM1")

display(
    paired_summary
)

print()
print(
    "08.24 fixed-policy comparison: COMPLETE"
)

In [ ]:
# ============================================================
# RUN — 08.25 Prefix-Aware Warm1/Warm2 Checkpoint Selection
# Purpose:
#   Test whether using Warm1 for 44b6 datasets and Warm2 for
#   6bba datasets improves Fold0 performance.
#
#   No parameter changes:
#     det = 0.995
#     high-confidence = 0.9975
#     K = 9
#     G2 p2 = 0.80
#     all other geometry/cleanup frozen
#
# Changes files: NO
# Runs training: NO
# Runs inference: NO
# Uses GT: YES — Fold0 model-selection evaluation
# Keep after running: YES — formal domain-selection evidence
# ============================================================

def evaluate_mixed_warm_policy():

    records = []
    rows = []

    for name in dataset_names:

        prefix = prefix_map[name]

        # ----------------------------------------
        # Domain-specific checkpoint selection
        # ----------------------------------------

        if prefix == "44b6":
            main_dir = WARM1_0995
            high_dir = WARM1_09975
            model_used = "warm1"

        elif prefix == "6bba":
            main_dir = WARM2_0995
            high_dir = WARM2_09975
            model_used = "warm2"

        else:
            raise ValueError(
                f"Unexpected prefix: {prefix}"
            )

        pred_g = load_graph(
            main_dir / f"{name}.geff"
        )

        high_g = load_graph(
            high_dir / f"{name}.geff"
        )

        gt_g = load_graph(
            TRAIN_DIR / f"{name}.geff"
        )

        main_keys = node_coordinate_keys(
            pred_g
        )

        high_keys = node_coordinate_keys(
            high_g
        )

        assert not (
            high_keys - main_keys
        )

        scale = np.asarray(
            open_dataset(
                TRAIN_DIR / name,
                load_image=False,
            ).scale,
            dtype=float,
        )

        # 1. Frozen G2 p2=.80
        apply_frozen_g2(
            pred_g,
            scale,
        )

        # 2. Frozen confidence K=9
        apply_low_conf_component_pruning(
            pred_g,
            high_keys,
            9,
        )

        # 3. Frozen cleanup
        apply_frozen_component_cleanup(
            pred_g,
            scale,
        )

        # 4. Locked official metric
        er = official_evaluate(
            pred_g,
            gt_g,
            scale=tuple(scale),
            max_distance=7.0,
        )

        recall = node_recall(
            pred_g,
            gt_g,
        )

        m = per_sample_metrics(
            er,
            float(
                estimated_map[name]
            ),
            recall,
        )

        records.append(m)

        rows.append({
            "dataset":
                name,

            "prefix":
                prefix,

            "model":
                model_used,

            "edge_jaccard":
                m["edge_jaccard"],

            "adj_edge_jaccard":
                m["adj_edge_jaccard"],

            "node_recall":
                m["node_recall"],

            "node_ratio":
                m["total_node_ratio"],
        })

    s = summarise(records)

    overall = {
        "edge_jaccard":
            s["edge_jaccard"],

        "adj_edge_jaccard":
            s["adj_edge_jaccard"],

        "division_jaccard":
            s["division_jaccard"],

        "node_recall":
            s["node_recall"],

        "median_node_ratio":
            float(
                np.median([
                    m["total_node_ratio"]
                    for m in records
                ])
            ),

        "score":
            s["score"],
    }

    return (
        overall,
        pd.DataFrame(rows),
        records,
    )


print(
    "=== 08.25 PREFIX-AWARE CHECKPOINT SELECTION ==="
)

mixed_overall, mixed_df, mixed_records = (
    evaluate_mixed_warm_policy()
)

comparison_0825 = pd.DataFrame([
    {
        "policy": "warm1_all",
        **warm1_overall,
    },
    {
        "policy": "warm2_all",
        **warm2_overall,
    },
    {
        "policy": "44b6_warm1__6bba_warm2",
        **mixed_overall,
    },
])

print()
print("OVERALL")

display(
    comparison_0825[
        [
            "policy",
            "edge_jaccard",
            "adj_edge_jaccard",
            "division_jaccard",
            "node_recall",
            "median_node_ratio",
            "score",
        ]
    ]
)

print()
print(
    "Mixed delta vs Warm2:",
    f"{mixed_overall['score'] - warm2_overall['score']:+.6f}",
)

print()
print(
    "Mixed delta vs previous best Warm1:",
    f"{mixed_overall['score'] - 0.760314:+.6f}",
)

print()
print(
    "08.25 prefix-aware checkpoint selection: COMPLETE"
)

In [ ]:
# ============================================================
# RUN — 08.26 Warm2 Confidence-K Recalibration
# Purpose:
#   Recalibrate only the confidence-aware component-size cutoff
#   K for the accepted Warm2 checkpoint.
#
# Frozen:
#   checkpoint = Warm2
#   det = 0.995
#   high-confidence = 0.9975
#   G2 p2_min = 0.80
#   all G2 geometry unchanged
#   singleton cleanup unchanged
#   2-node max displacement = 3.5 um
#
# Changes files: NO
# Runs training: NO
# Runs inference: NO
# Uses GT: YES — Fold0 post-processing calibration
# Keep after running: YES — formal calibration evidence
# ============================================================

assert "evaluate_fixed_policy" in globals()
assert "WARM2_0995" in globals()
assert "WARM2_09975" in globals()

ORIGINAL_K_0826 = CONF_COMPONENT_LIMIT

K_CANDIDATES = [
    0,
    2,
    4,
    9,
    19,
    np.inf,
]

results_0826 = []

print("=== 08.26 WARM2 K RECALIBRATION ===")
print()

try:

    for k in K_CANDIDATES:

        CONF_COMPONENT_LIMIT = k

        overall, prefix_df, dataset_df = (
            evaluate_fixed_policy(
                WARM2_0995,
                WARM2_09975,
            )
        )

        results_0826.append({
            "K":
                "all"
                if np.isinf(k)
                else int(k),

            **overall,

            "_prefix":
                prefix_df,

            "_dataset":
                dataset_df,
        })

finally:

    # Restore current baseline-v6 value.
    CONF_COMPONENT_LIMIT = ORIGINAL_K_0826


# ============================================================
# Overall
# ============================================================

overall_0826 = pd.DataFrame([
    {
        key: value
        for key, value in r.items()
        if not key.startswith("_")
    }
    for r in results_0826
])

overall_0826 = (
    overall_0826
    .sort_values(
        "score",
        ascending=False,
    )
    .reset_index(drop=True)
)

print("OVERALL")

display(
    overall_0826[
        [
            "K",
            "edge_jaccard",
            "adj_edge_jaccard",
            "division_jaccard",
            "node_recall",
            "median_node_ratio",
            "score",
            "conf_nodes_removed",
            "cleanup_nodes_removed",
        ]
    ]
)


# ============================================================
# Prefix
# ============================================================

prefix_rows = []

for result in results_0826:

    k = result["K"]

    prefix_df = result[
        "_prefix"
    ].copy()

    prefix_df.insert(
        0,
        "K",
        k,
    )

    prefix_rows.append(
        prefix_df
    )

prefix_0826 = (
    pd.concat(
        prefix_rows,
        ignore_index=True,
    )
    .sort_values(
        [
            "prefix",
            "score",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(drop=True)
)

print()
print("BY PREFIX")

display(
    prefix_0826[
        [
            "K",
            "prefix",
            "edge_jaccard",
            "adj_edge_jaccard",
            "division_jaccard",
            "node_recall",
            "median_node_ratio",
            "score",
        ]
    ]
)


# ============================================================
# Best
# ============================================================

best = overall_0826.iloc[0]

print()
print(
    "Best Warm2 K:",
    best["K"],
)

print(
    "Best score:",
    f"{best['score']:.6f}",
)

print(
    "Delta vs fixed Warm2 K=9:",
    f"{best['score'] - 0.767286:+.6f}",
)

print()
print(
    "CONF_COMPONENT_LIMIT restored to:",
    CONF_COMPONENT_LIMIT,
)

print()
print(
    "08.26 Warm2 K recalibration: COMPLETE"
)

In [ ]:
# ============================================================
# RUN — 08.27 Warm2 G2 p2 Recalibration
# Purpose:
#   Recalibrate only G2 p2_min for the accepted Warm2
#   checkpoint while keeping all other baseline settings frozen.
#
# Frozen:
#   checkpoint = Warm2
#   det = 0.995
#   high-confidence = 0.9975
#   confidence pruning K = 9
#   all G2 geometry except p2 unchanged
#   singleton cleanup unchanged
#   2-node max displacement = 3.5 um
#
# Changes files: NO
# Runs training: NO
# Runs inference: NO
# Uses GT: YES — Fold0 post-processing calibration
# Keep after running: YES — formal calibration evidence
# ============================================================

assert "evaluate_fixed_policy" in globals()

ORIGINAL_P2_0827 = float(
    G2_P2_MIN
)

# Limited sweep only.
# Includes original G2=.55 and Warm1 optimum=.80.
P2_CANDIDATES_0827 = [
    0.55,
    0.65,
    0.70,
    0.75,
    0.80,
    0.85,
]

results_0827 = []

print("=== 08.27 WARM2 G2 p2 RECALIBRATION ===")
print()

try:

    for p2 in P2_CANDIDATES_0827:

        G2_P2_MIN = float(p2)

        overall, prefix_df, dataset_df = (
            evaluate_fixed_policy(
                WARM2_0995,
                WARM2_09975,
            )
        )

        results_0827.append({
            "p2_min":
                float(p2),

            **overall,

            "_prefix":
                prefix_df,

            "_dataset":
                dataset_df,
        })

finally:

    G2_P2_MIN = ORIGINAL_P2_0827


# ============================================================
# Overall
# ============================================================

overall_0827 = pd.DataFrame([
    {
        key: value
        for key, value in r.items()
        if not key.startswith("_")
    }
    for r in results_0827
])

overall_0827 = (
    overall_0827
    .sort_values(
        "score",
        ascending=False,
    )
    .reset_index(drop=True)
)

print("OVERALL")

display(
    overall_0827[
        [
            "p2_min",
            "edge_jaccard",
            "adj_edge_jaccard",
            "division_jaccard",
            "node_recall",
            "median_node_ratio",
            "score",
        ]
    ]
)


# ============================================================
# Prefix
# ============================================================

prefix_rows = []

for result in results_0827:

    prefix_df = (
        result["_prefix"]
        .copy()
    )

    prefix_df.insert(
        0,
        "p2_min",
        result["p2_min"],
    )

    prefix_rows.append(
        prefix_df
    )

prefix_0827 = (
    pd.concat(
        prefix_rows,
        ignore_index=True,
    )
    .sort_values(
        [
            "prefix",
            "score",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(drop=True)
)

print()
print("BY PREFIX")

display(
    prefix_0827[
        [
            "p2_min",
            "prefix",
            "edge_jaccard",
            "adj_edge_jaccard",
            "division_jaccard",
            "node_recall",
            "median_node_ratio",
            "score",
        ]
    ]
)


# ============================================================
# Best candidate
# ============================================================

best = overall_0827.iloc[0]

print()
print(
    "Best Warm2 p2_min:",
    best["p2_min"],
)

print(
    "Best score:",
    f"{best['score']:.6f}",
)

print(
    "Delta vs Warm2 p2=.80:",
    f"{best['score'] - 0.767286:+.6f}",
)

print()
print(
    "G2_P2_MIN restored to:",
    G2_P2_MIN,
)

print()
print(
    "08.27 Warm2 p2 recalibration: COMPLETE"
)

In [ ]:
# ============================================================
# RUN — 08.28 Prefix-Specific G2 p2 Evaluation
# Purpose:
#   Test a domain-specific G2 p2 policy suggested by the
#   Warm2 coarse calibration:
#
#       44b6 -> p2_min = 0.70
#       6bba -> p2_min = 0.80
#
#   All other Warm2 baseline settings remain frozen.
#
# Changes files: NO
# Runs training: NO
# Runs inference: NO
# Uses GT: YES — Fold0 post-processing calibration
# Keep after running: YES — formal domain-policy evidence
# ============================================================

assert "apply_frozen_g2" in globals()
assert "apply_low_conf_component_pruning" in globals()
assert "apply_frozen_component_cleanup" in globals()

ORIGINAL_P2_0828 = float(
    G2_P2_MIN
)

PREFIX_P2 = {
    "44b6": 0.70,
    "6bba": 0.80,
}

records_0828 = []
rows_0828 = []

print(
    "=== 08.28 PREFIX-SPECIFIC G2 p2 EVALUATION ==="
)

try:

    for name in dataset_names:

        prefix = prefix_map[name]

        assert prefix in PREFIX_P2

        # ----------------------------------------
        # Select domain-specific p2 only
        # ----------------------------------------

        G2_P2_MIN = PREFIX_P2[
            prefix
        ]

        pred_g = load_graph(
            WARM2_0995
            / f"{name}.geff"
        )

        high_g = load_graph(
            WARM2_09975
            / f"{name}.geff"
        )

        gt_g = load_graph(
            TRAIN_DIR
            / f"{name}.geff"
        )

        main_keys = (
            node_coordinate_keys(
                pred_g
            )
        )

        high_keys = (
            node_coordinate_keys(
                high_g
            )
        )

        assert not (
            high_keys
            - main_keys
        )

        scale = np.asarray(
            open_dataset(
                TRAIN_DIR / name,
                load_image=False,
            ).scale,
            dtype=float,
        )

        # ----------------------------------------
        # 1. Exact frozen G2 implementation
        #    Only p2 differs by prefix.
        # ----------------------------------------

        apply_frozen_g2(
            pred_g,
            scale,
        )

        # ----------------------------------------
        # 2. Frozen K=9 confidence pruning
        # ----------------------------------------

        apply_low_conf_component_pruning(
            pred_g,
            high_keys,
            9,
        )

        # ----------------------------------------
        # 3. Frozen component cleanup
        # ----------------------------------------

        apply_frozen_component_cleanup(
            pred_g,
            scale,
        )

        # ----------------------------------------
        # 4. Locked official metric
        # ----------------------------------------

        er = official_evaluate(
            pred_g,
            gt_g,
            scale=tuple(scale),
            max_distance=7.0,
        )

        recall = node_recall(
            pred_g,
            gt_g,
        )

        m = per_sample_metrics(
            er,
            float(
                estimated_map[name]
            ),
            recall,
        )

        records_0828.append(
            m
        )

        rows_0828.append({
            "dataset":
                name,

            "prefix":
                prefix,

            "p2_min":
                PREFIX_P2[prefix],

            "edge_jaccard":
                m["edge_jaccard"],

            "adj_edge_jaccard":
                m["adj_edge_jaccard"],

            "node_recall":
                m["node_recall"],

            "node_ratio":
                m["total_node_ratio"],
        })

finally:

    G2_P2_MIN = (
        ORIGINAL_P2_0828
    )


# ============================================================
# Overall
# ============================================================

s = summarise(
    records_0828
)

mixed_0828 = {
    "policy":
        "44b6_p70__6bba_p80",

    "edge_jaccard":
        s["edge_jaccard"],

    "adj_edge_jaccard":
        s["adj_edge_jaccard"],

    "division_jaccard":
        s["division_jaccard"],

    "node_recall":
        s["node_recall"],

    "median_node_ratio":
        float(
            np.median([
                m["total_node_ratio"]
                for m in records_0828
            ])
        ),

    "score":
        s["score"],
}


comparison_0828 = pd.DataFrame([
    {
        "policy":
            "global_p80",

        "edge_jaccard":
            0.770197,

        "adj_edge_jaccard":
            0.763440,

        "division_jaccard":
            0.038462,

        "node_recall":
            0.953170,

        "median_node_ratio":
            0.056681,

        "score":
            0.767286,
    },

    mixed_0828,
])


# ============================================================
# Prefix reproduction
# ============================================================

df_0828 = pd.DataFrame(
    rows_0828
)

prefix_rows = []

for prefix in [
    "44b6",
    "6bba",
]:

    selected = [
        m
        for name, m in zip(
            dataset_names,
            records_0828,
        )
        if prefix_map[name]
        == prefix
    ]

    ps = summarise(
        selected
    )

    ratios = (
        df_0828.loc[
            df_0828["prefix"]
            == prefix,
            "node_ratio",
        ]
    )

    prefix_rows.append({
        "prefix":
            prefix,

        "p2_min":
            PREFIX_P2[prefix],

        "edge_jaccard":
            ps["edge_jaccard"],

        "adj_edge_jaccard":
            ps["adj_edge_jaccard"],

        "division_jaccard":
            ps["division_jaccard"],

        "node_recall":
            ps["node_recall"],

        "median_node_ratio":
            float(
                np.median(ratios)
            ),

        "score":
            ps["score"],
    })


prefix_0828 = pd.DataFrame(
    prefix_rows
)


# ============================================================
# Sanity gates
# ============================================================

score_44b6 = float(
    prefix_0828.loc[
        prefix_0828["prefix"]
        == "44b6",
        "score",
    ].iloc[0]
)

score_6bba = float(
    prefix_0828.loc[
        prefix_0828["prefix"]
        == "6bba",
        "score",
    ].iloc[0]
)

assert abs(
    score_44b6
    - 0.769925
) < 5e-6

assert abs(
    score_6bba
    - 0.768178
) < 5e-6


# ============================================================
# Result
# ============================================================

print()
print("OVERALL")

display(
    comparison_0828
)

print()
print("BY PREFIX")

display(
    prefix_0828
)

print()
print(
    "Mixed delta vs global p2=.80:",
    f"{mixed_0828['score'] - 0.767286:+.6f}",
)

print()
print(
    "G2_P2_MIN restored to:",
    G2_P2_MIN,
)

print()
print(
    "08.28 prefix-specific p2 evaluation: COMPLETE"
)

In [ ]:
# ============================================================
# RUN — 08.29 44b6 p2=.70 vs .80 Robustness Audit
# Purpose:
#   Verify whether the apparent advantage of p2=.70 on the
#   44b6 domain is broad and structurally meaningful, rather
#   than being driven by only one or two sparse division events.
#
# Frozen:
#   checkpoint = Warm2
#   det = 0.995
#   high-confidence = 0.9975
#   K = 9
#   all other G2 geometry unchanged
#   cleanup unchanged
#
# Changes files: NO
# Runs training: NO
# Runs inference: NO
# Uses GT: YES — Fold0 robustness evaluation
# Keep after running: YES — formal domain-policy evidence
# ============================================================

assert "apply_frozen_g2" in globals()
assert "apply_low_conf_component_pruning" in globals()
assert "apply_frozen_component_cleanup" in globals()

ORIGINAL_P2_0829 = float(
    G2_P2_MIN
)

P2_VARIANTS = {
    "p80": 0.80,
    "p70": 0.70,
}

TARGET_PREFIX = "44b6"

variant_records = {}
variant_rows = {}

print("=== 08.29 44b6 p2 ROBUSTNESS AUDIT ===")
print()

try:

    for label, p2 in P2_VARIANTS.items():

        G2_P2_MIN = float(p2)

        records = []
        rows = []

        for name in dataset_names:

            if prefix_map[name] != TARGET_PREFIX:
                continue

            pred_g = load_graph(
                WARM2_0995
                / f"{name}.geff"
            )

            high_g = load_graph(
                WARM2_09975
                / f"{name}.geff"
            )

            gt_g = load_graph(
                TRAIN_DIR
                / f"{name}.geff"
            )

            high_keys = (
                node_coordinate_keys(
                    high_g
                )
            )

            scale = np.asarray(
                open_dataset(
                    TRAIN_DIR / name,
                    load_image=False,
                ).scale,
                dtype=float,
            )

            # 1. G2 — only p2 differs
            apply_frozen_g2(
                pred_g,
                scale,
            )

            # 2. Frozen K=9
            apply_low_conf_component_pruning(
                pred_g,
                high_keys,
                9,
            )

            # 3. Frozen cleanup
            apply_frozen_component_cleanup(
                pred_g,
                scale,
            )

            # 4. Locked official metric
            er = official_evaluate(
                pred_g,
                gt_g,
                scale=tuple(scale),
                max_distance=7.0,
            )

            recall = node_recall(
                pred_g,
                gt_g,
            )

            m = per_sample_metrics(
                er,
                float(
                    estimated_map[name]
                ),
                recall,
            )

            records.append(m)

            rows.append({
                "dataset":
                    name,

                "edge_jaccard":
                    m["edge_jaccard"],

                "adj_edge_jaccard":
                    m["adj_edge_jaccard"],

                "node_recall":
                    m["node_recall"],

                "node_ratio":
                    m["total_node_ratio"],
            })

        variant_records[label] = records

        variant_rows[label] = (
            pd.DataFrame(rows)
            .set_index("dataset")
        )

finally:

    G2_P2_MIN = ORIGINAL_P2_0829


# ============================================================
# Aggregate summaries
# ============================================================

summary_rows = []

for label, p2 in P2_VARIANTS.items():

    s = summarise(
        variant_records[label]
    )

    summary_rows.append({
        "variant":
            label,

        "p2_min":
            p2,

        "edge_jaccard":
            s["edge_jaccard"],

        "adj_edge_jaccard":
            s["adj_edge_jaccard"],

        "division_jaccard":
            s["division_jaccard"],

        "division_tp":
            s["division_tp"],

        "division_fp":
            s["division_fp"],

        "division_fn":
            s["division_fn"],

        "node_recall":
            s["node_recall"],

        "score":
            s["score"],
    })

summary_0829 = pd.DataFrame(
    summary_rows
)


# ============================================================
# Paired dataset deltas: p70 - p80
# ============================================================

p80 = variant_rows["p80"]
p70 = variant_rows["p70"]

paired_0829 = pd.DataFrame(
    index=p80.index
)

for metric in [
    "edge_jaccard",
    "adj_edge_jaccard",
    "node_recall",
    "node_ratio",
]:

    paired_0829[
        f"delta_{metric}"
    ] = (
        p70[metric]
        - p80[metric]
    )

paired_0829 = (
    paired_0829
    .reset_index()
)

paired_summary_0829 = pd.DataFrame([{
    "datasets":
        len(paired_0829),

    "edgeJ_improved":
        int(
            (
                paired_0829[
                    "delta_edge_jaccard"
                ] > 0
            ).sum()
        ),

    "edgeJ_improved_rate":
        float(
            (
                paired_0829[
                    "delta_edge_jaccard"
                ] > 0
            ).mean()
        ),

    "adjEdgeJ_improved":
        int(
            (
                paired_0829[
                    "delta_adj_edge_jaccard"
                ] > 0
            ).sum()
        ),

    "adjEdgeJ_improved_rate":
        float(
            (
                paired_0829[
                    "delta_adj_edge_jaccard"
                ] > 0
            ).mean()
        ),

    "mean_delta_edgeJ":
        float(
            paired_0829[
                "delta_edge_jaccard"
            ].mean()
        ),

    "median_delta_edgeJ":
        float(
            paired_0829[
                "delta_edge_jaccard"
            ].median()
        ),

    "mean_delta_adjEdgeJ":
        float(
            paired_0829[
                "delta_adj_edge_jaccard"
            ].mean()
        ),

    "median_delta_adjEdgeJ":
        float(
            paired_0829[
                "delta_adj_edge_jaccard"
            ].median()
        ),

    "mean_delta_recall":
        float(
            paired_0829[
                "delta_node_recall"
            ].mean()
        ),
}])


# ============================================================
# Result
# ============================================================

print("AGGREGATE")
display(
    summary_0829
)

print()
print("PAIRED p70 - p80")
display(
    paired_summary_0829
)

print()
print(
    "G2_P2_MIN restored to:",
    G2_P2_MIN,
)

print()
print(
    "08.29 44b6 p2 robustness audit: COMPLETE"
)

MODEL
Warm2
SHA256:
31d209bfde1fd5bbf3b2a661cdb4a25bb72922b832d315ee5a75cc32f44656c1

DETECTION
det = .995
high confidence = .9975

CONFIDENCE PRUNING
K = 9

DIVISION
global p2 = .80

daughter separation >= 6 µm
angle >= 120°
max parent-daughter <= 8 µm
distance balance <= .50

CLEANUP
remove singleton components
remove 2-node components with displacement > 3.5 µm

ROBUST FOLD0 SCORE
0.767286

raw Edge Jaccard       = 0.770197
adjusted Edge Jaccard  = 0.763440
Division Jaccard       = 0.038462
node recall            = 0.953170
median node ratio      = +0.056681